# Concierge de Viajes Multiagente

> "Buscame vuelos a Bariloche, un hotel cerca del centro y que hacer el finde."

Un solo agente se marea con todo eso. Vamos a arrancar sin ningun framework, con una
clase de Python y un loop, y de ahi construimos hasta tener varios agentes que se
coordinan.

## Lo que vas a poder hacer al final

- Escribir un agente desde cero, sin framework, y entender exactamente que hace
- Saber que te da un framework cuando lo usas, porque primero lo hiciste a mano
- Ver por que un solo agente con muchas tools se equivoca, y como se arregla
- Coordinar varios agentes con un supervisor
- Entender que MCP es un protocolo de tools, nada mas

## Agenda

| Seccion | Tema | Min |
|---|---|---|
| 1 | El caso: tres pedidos en una sola frase | 3 |
| 2 | Un agente es un loop | 10 |
| 3 | Lo mismo, pero en LangGraph | 7 |
| 4 | Un agente con demasiadas tools | 7 |
| 5 | Lo partimos: un agente por capacidad | 8 |
| 6 | Quien le contesta al usuario | 9 |
| 7 | MCP, sin misticismo | 8 |
| 8 | Y si se organizan solos | 5 |
| 9 | Cierre | 3 |

Taller de Axel Sirota para Nerdearla.

**Todas las celdas corren tal como estan.** Los ejercicios son opcionales y no
bloquean nada: si te los salteas, el notebook sigue funcionando igual. Las
soluciones estan al final.


## Seccion 0: Preparamos el entorno

Dos celdas y arrancamos. La primera instala todo, la segunda carga las claves.

Axel les reparte un archivo `.env` con las claves adentro. En Colab, subilo con el
panel de la izquierda (el iconito de carpeta) y dejalo al lado del notebook. No hace
falta que toques nada mas.


In [ ]:
# Instalamos todo de una. En Colab esto tarda un minuto la primera vez.
# Las versiones estan clavadas a proposito: son las que probamos y sabemos que andan juntas.
!pip install -q \
    openai==3.16.2 \
    python-dotenv==1.2.3 \
    requests \
    langchain==1.4.2 \
    langchain-openai==1.6.3 \
    langgraph==1.2.12 \
    langgraph-supervisor==0.0.31 \
    langgraph-swarm==0.1.0 \
    mcp==2.2.0 \
    fastmcp==4.0.5

print("Listo. Si no viste ningun error rojo, seguimos.")


In [ ]:
# ===========================================================================
# Imports y claves (PARA CASA: es puro setup, no hay nada que aprender aca)
# ===========================================================================
import base64
import json
import os
import warnings

import requests
from dotenv import load_dotenv
from IPython.display import Image, display
from openai import OpenAI

# Estas dos librerias avisan de cosas que no podemos arreglar desde nuestro codigo
# (una de ellas llama internamente a una funcion vieja), asi que las silenciamos
# para que no nos ensucien la pantalla. Solo estas dos, nunca todos los warnings:
# si algo se rompe de verdad, queremos verlo.
from langgraph.warnings import LangGraphDeprecatedSinceV10

warnings.filterwarnings("ignore", category=LangGraphDeprecatedSinceV10)
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

# ===========================================================================
# Las claves salen del .env que reparte Axel (NARRAR)
# ===========================================================================
# Nunca escribas una clave en el notebook. Si la escribis, queda en el archivo,
# y el archivo lo compartis. El .env se queda en tu maquina y no se sube nunca.
# find_dotenv busca el .env en la carpeta de trabajo. En Colab, si subiste el
# archivo con el panel de la izquierda, queda en /content y lo encuentra solo.
# Si te dice "Missing credentials", el .env no esta donde el notebook lo busca.
load_dotenv()

MODEL = "gpt-4o-mini"          # barato y suficiente para todo el taller
client = OpenAI()              # toma OPENAI_API_KEY del entorno, sin pasarsela a mano
SERPER_KEY = os.environ["SERPER_API_KEY"]   # esta si la usamos explicitamente

# ===========================================================================
# Helper para ver los diagramas (PARA CASA)
# ===========================================================================
# Colab no dibuja Mermaid en las celdas de texto, asi que traemos el diagrama
# del repo y lo renderizamos como imagen.
DIAGRAMAS = (
    "https://raw.githubusercontent.com/"
    "axel-sirota/nerdearla_concierge_agent/main/diagrams"
)


def mostrar_diagrama(slug: str) -> None:
    """Trae un diagrama del repo y lo dibuja como imagen."""
    fuente = requests.get(f"{DIAGRAMAS}/{slug}.mmd", timeout=30)
    if fuente.status_code != 200:
        # Si el diagrama todavia no esta en el repo, avisamos en texto en lugar
        # de dibujar la pagina de error de GitHub.
        print(f"(El diagrama '{slug}' todavia no esta publicado.)")
        return
    codificado = base64.urlsafe_b64encode(fuente.text.encode()).decode()
    imagen = requests.get(f"https://mermaid.ink/img/{codificado}", timeout=30)
    display(Image(data=imagen.content))


print(f"Todo listo. Modelo: {MODEL}")


## Seccion 1: El caso

Tres pedidos distintos en una sola frase. Miremos el problema antes de escribir codigo.

<!-- /build-notebook llena esta seccion desde su plan. -->


## Seccion 2: Un agente es un loop

Python puro. Sin framework, sin nada importado que esconda el truco.

<!-- /build-notebook llena esta seccion desde su plan. -->


Antes de tocar un framework, quiero que veamos que hay abajo. Porque lo que sigue es la
parte que despues todos los frameworks te esconden, y si no la viste una vez a mano, el
framework te va a parecer magia.

Arranquemos por el problema. Le vamos a preguntar a `gpt-4o-mini` cuanto sale un vuelo a
Bariloche. Nada mas: el modelo solo, sin tools, sin nada.

Fijate en dos cosas cuando corra la celda que viene:

1. Te contesta con un numero.
2. Ese numero es inventado.

No esta mintiendo a proposito. El modelo no tiene forma de ir a mirar. Lo unico que sabe
hacer es predecir texto plausible, y un precio plausible es exactamente eso: plausible.


In [ ]:
# ===========================================================================
# EL PROBLEMA: el modelo solo, sin herramientas (NARRAR)
# ===========================================================================
# Ojo: no le pasamos ninguna tool. Es el modelo pelado, como lo usaste siempre.
# Le preguntamos un precio concreto, de esta semana, del mundo real.

respuesta = client.chat.completions.create(
    model=MODEL,
    messages=[
        # El system prompt define el personaje. Nada mas que eso por ahora.
        {"role": "system", "content": "Sos un agente de viajes argentino. Respondes corto."},
        {"role": "user", "content": "Cuanto sale un vuelo de Buenos Aires a Bariloche la "
                                    "semana que viene? Dame precio y aerolinea."},
    ],
    # PARA CASA: fijate que no hay parametro `tools` en esta llamada. Esa ausencia es
    # todo el punto de la celda: el modelo no tiene ninguna via para salir a buscar.
)

print(respuesta.choices[0].message.content)


### Lo que dijo el modelo, y lo que cuesta de verdad

En una corrida de prueba el modelo tiro un rango de ARS 15.000 a 30.000. Los precios
reales, buscados ese mismo minuto: **ARS 50.550, 57.186 y 70.457**. Se equivoco por dos a
cinco veces, y lo dijo con total seguridad.

Y aca esta la parte importante, porque es facil sacar la conclusion equivocada: el problema
no es que el numero este mal. El problema es que **el modelo no tiene forma de ir a
buscarlo**. Si le pegas el precio en el prompt arreglas esta pregunta y ninguna otra: no
sabes de antemano si te va a preguntar por vuelos, por hoteles o por que hacer el finde.

Lo que necesitamos es darle una forma de pedirnos que busquemos nosotros. Eso es un agente,
y el loop completo entra en un diagrama.

<!-- DIAGRAM: sequenceDiagram del loop del agente. Cuatro participantes: Vos, Tu codigo, Modelo, serper.dev. El usuario pide vuelos a Tu codigo; Tu codigo manda mensajes mas la lista de tools al Modelo; el Modelo devuelve tool_calls pidiendo buscar_vuelos; una Note sobre Tu codigo aclara que el modelo NO ejecuta nada; Tu codigo llama a serper.dev; serper devuelve resultados; Tu codigo manda el resultado al Modelo como mensaje role tool con su tool_call_id; el Modelo devuelve la respuesta final en texto; Tu codigo se la pasa al usuario. Es el diagrama mas importante del taller. -->


In [ ]:
# El diagrama mas importante del taller. Miralo dos veces.
mostrar_diagrama("agente-loop")


El diagrama muestra el viaje completo de una pregunta. Quedate con el paso del medio:
cuando el modelo contesta `tool_calls`, **no ejecuto nada**. Te devolvio un pedido escrito:
"llama a `buscar_vuelos` con estos argumentos". El que ejecuta sos vos. Siempre.


In [ ]:
# ===========================================================================
# LAYER 0: el wrapper de busqueda. Esta parte es solo algoritmica (NARRAR)
# ===========================================================================
SERPER_URL = "https://google.serper.dev/search"


def _serper(query: str, n: int = 5) -> list[dict]:
    """Consulta serper.dev y devuelve los resultados organicos crudos."""
    # PARA CASA: serper.dev es Google servido como JSON. Le mandamos un POST y nos
    # devuelve la misma pagina de resultados que verias en el navegador, ya parseada.
    respuesta = requests.post(
        SERPER_URL,
        # La clave va en un header, no en la URL: asi no queda en logs ni en el historial.
        headers={"X-API-KEY": SERPER_KEY,
                 "Content-Type": "application/json"},
        # gl="ar" y hl="es" son lo que hace que esto sirva para Argentina: resultados
        # geolocalizados y en español. Sin eso te llegan precios en dolares y paginas
        # en ingles.
        json={"q": query, "gl": "ar", "hl": "es", "num": n},
        # Timeout siempre. Un agente que se cuelga esperando una API es un agente colgado.
        timeout=20,
    )
    # Si la clave esta vencida o mal, esto explota ACA con un mensaje claro (403), en vez
    # de devolver una lista vacia y hacerte creer que Bariloche no existe.
    respuesta.raise_for_status()
    # "organic" son los resultados de siempre, los diez azules. El .get con [] por
    # defecto nos cubre de una busqueda sin resultados.
    return respuesta.json().get("organic", [])


def _normalizar(resultados: list[dict]) -> list[dict]:
    """Deja solo titulo, resumen y link de cada resultado."""
    # PARA CASA: el JSON de Google trae un monton de campos que no nos importan.
    # Nos quedamos con tres. Esto NO es para ahorrar tokens (medido: apenas un 6% menos):
    # es para que el modelo, y vos, lean algo prolijo en vez de un bloque de ruido.
    return [
        {
            "titulo": r["title"],
            # "snippet" casi siempre viene, pero no esta garantizado. El .get con ""
            # evita un KeyError en vivo por un resultado raro.
            "resumen": r.get("snippet", ""),
            "link": r["link"],
        }
        for r in resultados
    ]


print("Wrapper listo.")


In [ ]:
# ===========================================================================
# LAYER 1: las dos tools de verdad (NARRAR)
# ===========================================================================
# Estas dos funciones son las tools. No tienen nada especial: son funciones de Python
# comunes. Lo unico que las hace "tools" es que despues se las vamos a describir al
# modelo.


def buscar_vuelos(origen: str, destino: str) -> list[dict]:
    """Busca vuelos entre dos ciudades argentinas.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    # PARA CASA: el docstring no es decoracion. En un rato se lo vamos a copiar tal cual
    # al schema, y el modelo lo lee para decidir cual tool usar. Docstring vago, tool mal
    # elegida. Escribilos como si el lector fuera el modelo, porque lo es.
    return _normalizar(_serper(f"vuelos {origen} a {destino} precio"))


def buscar_hoteles(ciudad: str, zona: str = "centro") -> list[dict]:
    """Busca hoteles en una ciudad argentina.

    Args:
        ciudad: por ejemplo "Bariloche"
        zona: barrio o area, por defecto "centro"
    """
    # PARA CASA: `zona` tiene default. Si el modelo no lo manda, Python pone "centro".
    # Es un default de Python comun, el modelo no se entera ni le importa.
    return _normalizar(_serper(f"hoteles {zona} {ciudad} precio"))


# Probemos una, a mano, sin agente ni modelo de por medio. Esto es codigo y nada mas.
for vuelo in buscar_vuelos("Buenos Aires", "Bariloche")[:3]:
    print("-", vuelo["titulo"])
    print("  ", vuelo["resumen"][:90])


In [ ]:
# ===========================================================================
# EL SCHEMA A MANO: esto es lo que el modelo realmente ve (NARRAR)
# ===========================================================================
# Aca esta el truco entero, y lo escribimos a mano a proposito, una sola vez en todo
# el taller. Un framework te genera esto solo desde el docstring y los type hints,
# y esta bien que lo haga. Pero si nunca lo viste, no sabes que existe.
#
# El modelo NO recibe tu funcion de Python. No puede. Vive en otra computadora.
# Lo unico que recibe es esta descripcion en JSON: como se llama, para que sirve,
# y que argumentos acepta.

ESQUEMA_BUSCAR_VUELOS = {
    # "function" es el unico tipo que vamos a usar hoy.
    "type": "function",
    "function": {
        # El nombre TIENE que coincidir exactamente con la clave del dict de tools que
        # le pasemos al agente. Si no coincide, el dispatch no encuentra la funcion.
        "name": "buscar_vuelos",
        # Esta linea es la que el modelo lee para elegir. Es el docstring, copiado.
        # Es el campo mas importante del schema y el que mas gente escribe al apuro.
        "description": "Busca vuelos entre dos ciudades argentinas.",
        # Los argumentos, en JSON Schema.
        "parameters": {
            "type": "object",
            "properties": {
                "origen": {"type": "string",
                           "description": "ciudad de salida, por ejemplo Buenos Aires"},
                "destino": {"type": "string",
                            "description": "ciudad de llegada, por ejemplo Bariloche"},
            },
            # Lo que el modelo esta obligado a mandar. Lo que no este aca es opcional.
            "required": ["origen", "destino"],
        },
    },
}

# Miralo impreso. Son 24 lineas de JSON. Eso es toda la "magia" de las tools.
print(json.dumps(ESQUEMA_BUSCAR_VUELOS, indent=2, ensure_ascii=False))


In [ ]:
# ===========================================================================
# LA CLASE, PARTE 1: que se guarda un agente (NARRAR)
# ===========================================================================
class AgenteVuelos:
    """Un agente es esto: un loop que le pregunta al modelo que tool usar."""

    def __init__(self, tools: dict[str, callable], system: str, model: str = MODEL):
        # `tools` es un diccionario de nombre a funcion de Python:
        #     {"buscar_vuelos": buscar_vuelos}
        # Esta es la tabla de dispatch. Cuando el modelo pida "buscar_vuelos",
        # buscamos esa clave aca y llamamos a la funcion. No hay nada mas que eso.
        self.tools = tools
        self.system = system
        self.model = model
        # PARA CASA: la conversacion es una lista de Python comun. No es un objeto
        # con estado oculto ni un "contexto" magico. Es una lista, y la vamos a hacer
        # crecer nosotros a mano. Podes imprimirla cuando quieras: lo vas a hacer en
        # un ejercicio al final de esta seccion.
        self.messages: list[dict] = [{"role": "system", "content": system}]

    def _tool_schemas(self) -> list[dict]:
        """Traduce las funciones de Python al formato que espera OpenAI."""
        # Devolvemos la lista de schemas que armamos a mano en la celda anterior.
        # En un framework esta funcion no existe: se genera sola. Aca la vemos.
        return [ESQUEMA_BUSCAR_VUELOS]


print("Parte 1 lista: ya sabe que guardar. Todavia no sabe hacer nada.")


In [ ]:
# ===========================================================================
# LA CLASE, PARTE 2: el loop. Esto es todo lo que un agente es (NARRAR)
# ===========================================================================
# Truco de notebook: reabrimos la clase heredando de si misma, solo para poder mostrar
# el loop en su propia celda. En un archivo .py los dos bloques serian una sola clase.
class AgenteVuelos(AgenteVuelos):

    def run(self, pregunta: str, max_turns: int = 5) -> str:
        """El loop. Esto es todo lo que un agente es."""
        # La pregunta del usuario entra a la conversacion como un mensaje mas.
        self.messages.append({"role": "user", "content": pregunta})

        turn = 0
        # `max_turns` es el freno de mano. Sin un techo duro no tenes una garantia,
        # tenes una esperanza: si el modelo se obstina en pedir tools, el loop no para.
        # En produccion esto se llama "bounded execution" y es obligatorio.
        while turn < max_turns:
            turn += 1

            # --- PASO 1: le preguntamos al modelo, con la lista de tools adjunta -----
            resp = client.chat.completions.create(
                model=self.model,
                messages=self.messages,          # TODA la conversacion, cada vez
                tools=self._tool_schemas(),      # el JSON de la celda anterior
            )
            msg = resp.choices[0].message

            # --- PASO 2: el modelo pidio una tool, o ya contesto? -------------------
            if not msg.tool_calls:
                # No pidio nada mas: lo que hay en `content` es la respuesta final.
                # Aca sale el loop. Una pregunta con una tool tarda dos vueltas:
                # en la primera pide la tool, en la segunda escribe la respuesta.
                self.messages.append({"role": "assistant", "content": msg.content})
                return msg.content

            # --- PASO 3: appendear el mensaje del modelo ANTES que nada --------------
            # OBLIGATORIO y facil de olvidar. Si mandas la respuesta de la tool sin haber
            # appendeado primero este mensaje, la API te devuelve 400:
            #   "messages with role 'tool' must be a response to a preceeding message
            #    with 'tool_calls'"
            # `exclude_none=True` saca los campos vacios (como content, que viene en None
            # cuando el modelo pide tools) que la API no quiere recibir.
            self.messages.append(msg.model_dump(exclude_none=True))

            # --- PASO 4: NOSOTROS ejecutamos. El modelo nunca ejecuto nada -----------
            # El `for` no es decoracion: el modelo puede pedir VARIAS tools en un solo
            # mensaje, y lo hace seguido. Y hay que contestarle a CADA tool_call_id, si
            # te salteas uno la API devuelve 400:
            #   "must be followed by tool messages responding to each 'tool_call_id'"
            for call in msg.tool_calls:
                nombre = call.function.name
                # Los argumentos vienen como STRING con JSON adentro, no como dict.
                args = json.loads(call.function.arguments)
                print(f"   [turno {turn}] el modelo pidio: {nombre}({args})")

                # La tabla de dispatch en accion. Una busqueda en un dict y una llamada.
                # Esta linea es el corazon del agente, y es una linea de Python comun.
                resultado = self.tools[nombre](**args)

                # --- PASO 5: devolvemos el resultado como un mensaje mas -------------
                # `content` tiene que ser un STRING. Si le pasas la lista de dicts tal
                # cual, la API te devuelve 400 con un error medio cifrado:
                #   "Missing required parameter: 'messages[N].content[0].type'"
                # ensure_ascii=False para que los acentos se lean, aca y en pantalla.
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,   # ata esta respuesta a ESE pedido
                    "content": json.dumps(resultado, ensure_ascii=False),
                })
            # Volvemos al while: el modelo ahora ve el resultado y decide otra vez.

        # Si llegamos aca, se agotaron los turnos sin respuesta final. Devolvemos un
        # string igual, porque run() promete devolver un string siempre.
        return "Me quede sin turnos."


print("Parte 2 lista: ahora si es un agente.")


In [ ]:
# ===========================================================================
# A CORRERLO (NARRAR)
# ===========================================================================
# Le damos UNA sola tool, aunque `buscar_hoteles` ya existe y esta ahi al lado.
# Eso es a proposito, y en un rato vamos a ver por que.
agente_vuelos = AgenteVuelos(
    tools={"buscar_vuelos": buscar_vuelos},
    system=("Sos un agente de viajes argentino. Usa las tools que tengas para buscar "
            "datos reales antes de contestar. Respondes corto y en español rioplatense."),
)

# La pregunta de siempre, la de Bariloche.
respuesta_agente = agente_vuelos.run("Buscame vuelos de Buenos Aires a Bariloche.")

print()
print(respuesta_agente)
print()
# PARA CASA: cinco mensajes, y cada uno es un paso del diagrama:
# system (el personaje), user (la pregunta), assistant (pidio la tool),
# tool (lo que encontramos), assistant (la respuesta final).
print("Mensajes en la conversacion:", len(agente_vuelos.messages))
print("Roles:", [m["role"] for m in agente_vuelos.messages])


### Lo que acabamos de construir

Contemos las lineas que importan: un `while`, una llamada a la API, un `if` que pregunta si
hay `tool_calls`, un `for` que despacha, y una lista a la que le hacemos `append`. Eso es
un agente completo y funcionando.

Y ahora mira los imports de esta seccion: `json`, `os`, `requests`, `openai`. **No hay ni
un framework.** Ninguna de las cinco lineas de arriba nos la dio una libreria de agentes.

<!-- DIAGRAM: graph LR con la anatomia de la clase AgenteVuelos. Un subgraph AgenteVuelos con el while turn menor a max_turns en el centro, que lleva a chat.completions.create, que lleva a un rombo de decision hay tool_calls; la rama no va a return contenido; la rama si va a dispatch self.tools, de ahi a append role tool, y de ahi vuelve al while cerrando el ciclo. Afuera del subgraph, dos cajas punteadas: OpenAI conectada a chat.completions.create, y serper.dev conectada al dispatch. Conecta el codigo que acaban de ver con el Diagrama 2. -->


In [ ]:
# El mismo loop del diagrama anterior, pero dibujado como codigo.
mostrar_diagrama("agente-anatomia")


El diagrama toma el mismo loop del Diagrama 2 y lo dibuja como codigo: donde esta el
`while`, donde el `if`, y que dos cosas viven **afuera** de tu proceso. Las dos cajas
punteadas son las unicas piezas que no controlas: la API de OpenAI y serper.dev.

Tres cosas para llevarse de esta seccion:

1. **El modelo nunca ejecuta nada.** Te devuelve un pedido. El que ejecuta sos vos, en el
   `for`, con un dict y una llamada a funcion.
2. **La conversacion es una lista.** Le hacemos `append` a mano, en un orden que la API
   exige: primero el mensaje del modelo, despues una respuesta por cada `tool_call_id`.
3. **El loop termina** porque el modelo deja de pedir tools, o porque se acaban los turnos.
   Nunca dejes un agente sin techo.

Todo esto que acabamos de escribir a mano, un framework ya lo tiene resuelto. Vamos a ver
cuanto codigo nos borra, y sobre todo: ahora sabes exactamente que codigo te esta borrando.


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Ninguno de estos es necesario para seguir el taller. El notebook corre igual si los
# salteas. Las soluciones estan todas al final del notebook.

# EJERCICIO 1 (opcional, para despues)
# Agregale `buscar_hoteles` al agente, que ya esta definida y sin usar.
# Son dos cambios: un schema nuevo en `_tool_schemas` (copia el de vuelos y cambiale
# nombre, description y parameters), y una clave mas en el dict `tools`.
# Despues pedile: "vuelos de Buenos Aires a Bariloche y un hotel cerca del centro".
# Pista: fijate si el modelo pide las dos tools en el mismo turno o en turnos separados.
# Solucion al final del notebook.

# EJERCICIO 2 (opcional, para despues)
# Imprimi `agente_vuelos.messages` y leete la conversacion completa, mensaje por mensaje.
# Es una lista de Python comun, no hay nada escondido ahi adentro.
# Pista: para que se lea, imprimi el rol y los primeros 60 caracteres de cada mensaje.
# Ojo con el mensaje del assistant: despues de model_dump es un dict, asi que las
# tool_calls se leen con corchetes, m["tool_calls"], no con punto.
# Solucion al final del notebook.

# EJERCICIO 3 (opcional, para despues)
# Corre el agente con `max_turns=1` y mira que pasa.
# Pista: el modelo pide la tool en el turno 1, asi que nunca llega a escribir la respuesta
# final. Fijate que devuelve `run()` y como quedo la lista de mensajes, cortada al medio.
# Esto es exactamente para lo que existe el freno de mano.
# Solucion al final del notebook.


## Seccion 3: Lo mismo, pero en LangGraph

El mismo agente, migrado. Ahora si podes leer el framework, porque ya lo hiciste a mano.

<!-- /build-notebook llena esta seccion desde su plan. -->


Recien escribimos un agente sin framework. Funciona. Y funciona porque escribimos, a mano,
todo esto:

- el `while` con el contador de turnos
- el `if not msg.tool_calls` para saber cuando parar
- el `json.loads` de los argumentos que manda el modelo
- el diccionario para despachar la funcion por nombre
- y **24 lineas de JSON Schema** para describir UNA sola tool

Ese JSON es el que mas duele. Fijate que la descripcion que pusimos a mano ("ciudad de
salida, por ejemplo Buenos Aires") ya estaba escrita en el docstring de `buscar_vuelos`.
La escribimos dos veces: una para los humanos y una para el modelo.

Ahora hacemos el mismo agente en LangGraph. Mismo `buscar_vuelos`, misma pregunta, mismos
datos de serper. La gracia no es que quede mas corto. La gracia es que **ahora podes leer
el framework**, porque hace diez minutos escribiste lo que hay adentro.


In [ ]:
# ===========================================================================
# EL SCHEMA A MANO CONTRA EL SCHEMA GENERADO (NARRAR)
# ===========================================================================
# Arrancamos por lo que mas trabajo nos dio: el JSON Schema.
# LangChain trae una funcion que convierte una funcion de Python en una tool.
# Le pasamos parse_docstring=True a proposito: sin eso, LangChain manda el docstring
# entero como una sola descripcion y PIERDE la descripcion de cada argumento.
# Con eso puesto, lee el bloque "Args:" y arma el mismo schema que tipeamos a mano.
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

# OJO: no redefinimos buscar_vuelos. Es la misma funcion de la seccion anterior, envuelta.
vuelos_tool = tool(buscar_vuelos, parse_docstring=True)

# El schema que tipeamos a mano, sacado del agente que ya corrimos.
a_mano = AgenteVuelos({"buscar_vuelos": buscar_vuelos}, "comparacion")._tool_schemas()[0]

# El schema que LangChain genero leyendo el docstring en castellano.
generado = convert_to_openai_tool(vuelos_tool)

print("=== LO QUE ESCRIBIMOS A MANO ===")
print(json.dumps(a_mano, indent=2, ensure_ascii=False))
print()
print("=== LO QUE GENERO LANGCHAIN DEL DOCSTRING ===")
print(json.dumps(generado, indent=2, ensure_ascii=False))
print()

# PARA CASA: no pedimos que sean identicos caracter por caracter. Los textos salen del
# docstring, asi que son mas completos. Lo que importa es que el modelo recibe la misma
# forma: mismo nombre, mismos argumentos, mismos obligatorios.
print("mismo nombre de tool:",
      a_mano["function"]["name"] == generado["function"]["name"])
print("mismos argumentos:",
      set(a_mano["function"]["parameters"]["properties"])
      == set(generado["function"]["parameters"]["properties"]))
print("mismos obligatorios:",
      a_mano["function"]["parameters"]["required"]
      == generado["function"]["parameters"]["required"])


### El loop que escribimos, y el mismo loop en LangGraph

Las dos mitades van juntas en pantalla. A la izquierda lo de la seccion anterior, a la
derecha lo que vamos a construir ahora. Las cajas se corresponden una a una.

<!-- DIAGRAM: 4 - side-by-side. A la izquierda el loop a mano del Beat 1 con las etiquetas de su propio codigo (pregunta, client.chat.completions.create, if not msg.tool_calls return, for call in msg.tool_calls, self.tools[nombre](**args), while turn < max_turns). A la derecha el grafo que LangGraph compila de verdad, extraido con get_graph(): nodos __start__, model, tools, __end__; aristas __start__ -> model directa, model -> tools condicional, model -> __end__ condicional, tools -> model el loop. Las cajas de un lado se corresponden una a una con las del otro. Las dos mitades TIENEN que estar en pantalla juntas. -->


In [ ]:
# Las dos mitades juntas: a la izquierda lo nuestro, a la derecha el grafo.
mostrar_diagrama("langgraph")


Los nodos del grafo no son invento nuestro: se llaman `model` y `tools`, y los vas a ver
impresos con esos nombres cuando corramos el agente en un rato.


In [ ]:
# ===========================================================================
# EL AGENTE, EN CINCO LINEAS (NARRAR)
# ===========================================================================
# create_agent es la forma actual de armar un agente en LangChain 1.x.
# Si viste tutoriales con create_react_agent de langgraph.prebuilt: esa quedo
# deprecada, te tira un warning en pantalla y apunta aca. Usamos la actual.
from langchain.agents import create_agent

# El mismo prompt para los dos agentes, para que la comparacion sea honesta.
SYSTEM_VUELOS = (
    "Sos un asistente de viajes argentino. Contesta corto y en español rioplatense."
)

# Y esto es TODO el agente. Lo que antes fueron 28 lineas de clase.
agente_vuelos_lg = create_agent(
    # "openai:" + el modelo. Reusamos la constante MODEL para que el notebook
    # tenga un solo lugar donde cambiar de modelo.
    model=f"openai:{MODEL}",
    # La lista de tools. Agregar otra tool es agregar un elemento a esta lista:
    # nada de JSON a mano. Acordate de esto en la seccion que viene.
    tools=[vuelos_tool],
    # Lo que antes era self.messages[0]. Ojo: en LangGraph el system prompt
    # NO vive dentro de la lista de mensajes, va por separado.
    system_prompt=SYSTEM_VUELOS,
)

# No es un servicio ni un runtime raro: es un objeto de Python con .invoke().
print("tipo:", type(agente_vuelos_lg).__name__)
print("nodos del grafo:", list(agente_vuelos_lg.nodes))


In [ ]:
# ===========================================================================
# EL LOOP, POR DENTRO (NARRAR)
# ===========================================================================
# Hasta aca te pedimos que creas que adentro hay un loop. Ahora lo miramos.
# .stream() nos entrega el estado despues de cada nodo, asi que podemos imprimir
# el recorrido: es el diagrama del loop, narrado por el framework mismo.
PREGUNTA_VUELOS = "Buscame vuelos de Buenos Aires a Bariloche"

for paso in agente_vuelos_lg.stream(
    {"messages": [{"role": "user", "content": PREGUNTA_VUELOS}]}
):
    # paso es un dict con UNA clave: el nombre del nodo que acaba de correr.
    for nodo, estado_nodo in paso.items():
        ultimo = estado_nodo["messages"][-1]
        if getattr(ultimo, "tool_calls", None):
            # El nodo model decidio que necesita una tool. Esto es exactamente
            # el msg.tool_calls que leiamos a mano antes.
            pedido = ultimo.tool_calls[0]
            print(f"  [{nodo}] el modelo pide: {pedido['name']}({pedido['args']})")
        elif type(ultimo).__name__ == "ToolMessage":
            # El nodo tools ejecuto la funcion y metio el resultado como mensaje.
            # Esto es nuestro self.messages.append({"role": "tool", ...}).
            print(f"  [{nodo}] la tool devolvio {len(ultimo.content)} caracteres")
        else:
            # El modelo contesto sin pedir nada mas: el loop termina.
            # Esto es nuestro if not msg.tool_calls: return msg.content.
            print(f"  [{nodo}] el modelo contesta, sin pedir mas tools")


### Caja por caja

| Lo que escribimos a mano | Como se llama en LangGraph |
|---|---|
| `client.chat.completions.create(...)` | el nodo `model` |
| `for call in msg.tool_calls:` mas `self.tools[nombre](**args)` | el nodo `tools` |
| `if not msg.tool_calls: return msg.content` | la arista condicional `model -> __end__` |
| el `while` que vuelve a preguntar | la arista `tools -> model` |
| `self.messages` que iba creciendo | el estado del grafo, la clave `messages` |
| las 24 lineas de JSON Schema | el docstring, parseado por `tool(...)` |
| `max_turns=5` | `recursion_limit`, o `ModelCallLimitMiddleware(run_limit=5)` |
| `self.messages[0]` con el system | el argumento `system_prompt`, que va aparte |

No hay ninguna fila sin par. Eso es lo que compramos: no un concepto nuevo, el mismo
concepto con el plomero ya escrito.


In [ ]:
# ===========================================================================
# LOS DOS AGENTES, LA MISMA PREGUNTA (NARRAR)
# ===========================================================================
# La prueba honesta: la misma pregunta a los dos agentes, uno al lado del otro.
# Los dos usan la misma funcion buscar_vuelos y le pegan al mismo serper.

# El de la seccion anterior: devuelve directamente el string final.
respuesta_a_mano = agente_vuelos.run(PREGUNTA_VUELOS)

# El de LangGraph: devuelve el estado completo, con TODOS los mensajes.
# La respuesta final es el contenido del ultimo mensaje.
estado = agente_vuelos_lg.invoke(
    {"messages": [{"role": "user", "content": PREGUNTA_VUELOS}]}
)
respuesta_langgraph = estado["messages"][-1].content

print("=== A MANO ===")
print(respuesta_a_mano[:400])
print()
print("=== LANGGRAPH ===")
print(respuesta_langgraph[:400])
print()

# PARA CASA: si te interesa ver el viaje completo, esta todo en el estado.
# Fijate que el system prompt NO aparece: LangGraph lo guarda aparte.
print("mensajes que quedaron en el estado:",
      [type(m).__name__ for m in estado["messages"]])


### Que compramos y que nos esconde

Compramos el plomero: 28 lineas de clase se volvieron 5, el JSON Schema sale del
docstring, y el estado de la conversacion lo maneja el grafo.

Tambien compramos cosas que no vamos a usar hoy y que valen para cuando esto pase de demo
a algo real: guardar la conversacion entre llamadas con un `checkpointer`, cortar el agente
con `ModelCallLimitMiddleware`, resumir el historial cuando se hace largo, o pedir
aprobacion humana antes de ejecutar una tool sensible.

Y nos esconde el loop. Por eso lo escribimos primero. Cuando un agente en LangGraph se te
va de las manos, lo que tenes que mirar es lo mismo que mirabamos a mano: que tool pidio el
modelo, con que argumentos, y que le devolvio la tool.

Hay una cosa mas, y es la que nos mete en problemas en la proxima seccion. Agregar una tool
ahora sale **una linea**: un elemento mas en `tools=[...]`. Cuando algo sale tan barato,
uno agrega. Y despues agrega otra. Vamos a ver que pasa cuando un solo agente termina con
seis.


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Las soluciones estan todas al final del notebook.

# EJERCICIO 4 (opcional, para despues)
# Arma otro agente igual a agente_vuelos_lg pero cambiandole el system_prompt
# por "Contesta en una sola linea, sin listas". Hacele la misma pregunta y compara.
# Es el unico parametro que tocaste y la respuesta cambia entera: el prompt es
# parte del agente, no un adorno.
# Solucion al final del notebook.

# EJERCICIO 5 (opcional, para despues)
# Imprimi agente_vuelos_lg.get_graph().draw_mermaid() y busca en el texto que sale
# la linea "tools --> model". Esa flecha es el while de la seccion anterior.
# No necesitas instalar nada para esto.
# Solucion al final del notebook.

# EJERCICIO 6 (opcional, para despues, y el mas interesante de los tres)
# Envolve buscar_vuelos SIN parse_docstring=True, convertilo con convert_to_openai_tool
# y compara el schema con el de la celda de arriba. Vas a ver que las descripciones de
# cada argumento desaparecen y que el docstring entero queda apretado en un solo campo.
# Moraleja: el framework no adivina nada, te parsea el docstring, y le podes pedir que
# lo parsee mal.
# Solucion al final del notebook.


## Seccion 4: Un agente con demasiadas tools

Le damos seis tools a un solo agente y lo vemos elegir mal, en vivo.

<!-- /build-notebook llena esta seccion desde su plan. -->


Hasta aca tenemos dos tools que andan: `buscar_vuelos` y `buscar_hoteles`. Y tenemos el
loop, que lo escribimos nosotros, asi que sabemos exactamente que hace.

Ahora viene el pedido completo:

> "Buscame vuelos a Bariloche, un hotel cerca del centro y que hacer el finde."

Que harias vos? Lo mismo que hacemos todos la primera vez: un solo agente, y le colgamos
todas las tools que hacen falta. Es una linea de codigo. Vamos a hacerlo, y vamos a ver
por que es una trampa.


### Como llegamos a seis tools

Nadie arranca con seis tools. Se llega. Fijate la secuencia, que la vas a reconocer:

1. Arrancas con `buscar_vuelos`. Anda.
2. Un usuario pide algo mas barato. Alguien agrega `buscar_vuelos_baratos`. Nadie borra
   la primera, porque hay codigo que la usa.
3. Lo mismo del otro lado: `buscar_hoteles`, y despues `buscar_alojamiento`.
4. Aparece `buscar_transporte`, que nadie sabe bien si son vuelos o micros.
5. Y `buscar_actividades`, que es la unica que no se pisa con nada.

Seis tools. Dos pares que hacen lo mismo. Y el agente sigue siendo el mismo agente.


In [ ]:
# ===========================================================================
# LAS SEIS TOOLS (NARRAR)
# ===========================================================================
# Dos ya las tenemos de antes: buscar_vuelos y buscar_hoteles.
# Aca definimos las otras cuatro, y todas usan el mismo wrapper _serper de siempre.
#
# Prestá atencion a los docstrings. El modelo elige la tool LEYENDO ESTO.
# Para el modelo, el docstring no es documentacion: es la interfaz.


def buscar_vuelos_baratos(origen: str, destino: str) -> list[dict]:
    """Busca vuelos entre dos ciudades argentinas, con los mejores precios.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    # Se pisa con buscar_vuelos A PROPOSITO. Y fijate que el docstring agrega
    # "con los mejores precios": esa frase de mas es toda la diferencia, y es la
    # que despues nos va a costar caro.
    return _normalizar(_serper(f"vuelos baratos {origen} {destino}"))


def buscar_alojamiento(ciudad: str, zona: str = "centro") -> list[dict]:
    """Busca alojamiento en una ciudad argentina, con los mejores precios.

    Args:
        ciudad: por ejemplo "Bariloche"
        zona: barrio o area, por defecto "centro"
    """
    # El gemelo de buscar_hoteles. Mismo truco: "con los mejores precios".
    return _normalizar(_serper(f"alojamiento barato {zona} {ciudad}"))


def buscar_transporte(origen: str, destino: str) -> list[dict]:
    """Busca transporte entre dos ciudades argentinas.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    # Esta es la vaga: "transporte" puede ser un vuelo, un micro o un remis.
    # En un sistema real es la tool que nadie se anima a borrar.
    return _normalizar(_serper(f"como llegar {origen} a {destino}"))


def buscar_actividades(ciudad: str) -> list[dict]:
    """Busca actividades y paseos en una ciudad argentina.

    Args:
        ciudad: por ejemplo "Bariloche"
    """
    # La unica de las seis que no se pisa con ninguna otra. Acordate de esta,
    # porque al final va a ser la unica que el agente elige bien.
    return _normalizar(_serper(f"que hacer en {ciudad} fin de semana"))


# El catalogo completo: las dos de siempre mas las cuatro nuevas.
# Las claves son los nombres que ve el modelo; los valores, las funciones que
# nosotros vamos a ejecutar cuando el modelo las pida.
TOOLS_MUCHAS = {
    "buscar_vuelos": buscar_vuelos,                    # la original
    "buscar_vuelos_baratos": buscar_vuelos_baratos,    # gemela de la anterior
    "buscar_hoteles": buscar_hoteles,                  # la original
    "buscar_alojamiento": buscar_alojamiento,          # gemela de la anterior
    "buscar_transporte": buscar_transporte,            # vaga
    "buscar_actividades": buscar_actividades,          # la unica clara
}

print(f"El agente va a tener {len(TOOLS_MUCHAS)} tools:")
for nombre in TOOLS_MUCHAS:
    print(f"  - {nombre}")


### El catalogo, dibujado

<!-- DIAGRAM: un agente en el centro con las 6 tools colgando. buscar_vuelos y buscar_vuelos_baratos resaltadas como par que se pisa; buscar_hoteles y buscar_alojamiento resaltadas como el otro par. buscar_transporte marcada como vaga. buscar_actividades la unica limpia. -->


In [ ]:
# Mira las dos parejas resaltadas antes de seguir.
mostrar_diagrama("muchas-tools-catalogo")


Un agente, seis tools. Los dos pares resaltados son los que hacen lo mismo con dos nombres
distintos.

### 🎲 Antes de correrlo: apostemos

Pará aca un segundo. No corras la celda todavia.

El pedido va a ser el de siempre: vuelos a Bariloche, hotel en el centro, y que hacer el
finde. El agente tiene estas seis tools y un system prompt que le pide priorizar el precio,
que es lo que le pediria cualquier producto real.

**Que pensas que va a pasar?**

1. Elige bien las tres tools que corresponden.
2. Se cuelga y no llama ninguna.
3. Contesta cualquier cosa, un delirio.
4. Contesta perfecto, y usa las tools equivocadas.

Quedate con tu numero. Ahora si, corremos.


In [ ]:
# ===========================================================================
# LA CLASE: es LA MISMA de antes, heredada (NARRAR)
# ===========================================================================
# Aca esta la mitad del argumento de todo el taller: es LA MISMA CLASE.
# No cambiamos el loop. No cambiamos nada de como despachamos tool_calls.
# Heredamos y listo.
#
# Lo unico que agregamos es que se acuerde de que tools uso, para poder mostrarlo
# despues. Eso es instrumentacion nuestra, no es parte del agente.
#
# Que quede clarisimo: si esto falla, no falla el framework (no hay framework) ni
# falla el loop (es el mismo que ya vimos andar). Falla el catalogo de tools.

class AgenteVuelosMuchasTools(AgenteVuelos):
    """El mismo agente de antes, pero con seis tools en vez de una."""

    def __init__(self, tools: dict, system: str, model: str = MODEL):
        super().__init__(tools=tools, system=system, model=model)
        # Lista donde vamos anotando cada tool que el modelo pide, en orden.
        # Sin esto la unica forma de saber que eligio es leerle la mente.
        self.tools_usadas: list[str] = []

    def _tool_schemas(self) -> list[dict]:
        """Los schemas de las seis tools, generados desde los docstrings."""
        # PARA CASA: en la seccion anterior vimos que LangChain puede generar el
        # schema leyendo el docstring. Lo reusamos aca para no tipear seis JSON
        # a mano. El punto de esta seccion no es el schema, es la eleccion.
        return [convert_to_openai_tool(tool(f, parse_docstring=True))
                for f in self.tools.values()]

    def run(self, pregunta: str, max_turns: int = 5) -> str:
        """El loop, otra vez. Igual al de antes, con un print de mas."""
        self.messages.append({"role": "user", "content": pregunta})
        turno = 0
        while turno < max_turns:
            turno += 1
            resp = client.chat.completions.create(
                model=self.model,
                # temperature=1.0 es lo que pide el guion. Lo hablamos en la celda
                # de abajo, porque no es lo que vos crees que es.
                temperature=1.0,
                messages=self.messages,
                tools=self._tool_schemas(),
            )
            msg = resp.choices[0].message

            # Sin tool_calls el modelo ya tiene la respuesta final. Salimos.
            if not msg.tool_calls:
                return msg.content

            # Guardamos el mensaje del asistente TAL CUAL vino, con los tool_calls
            # adentro. Si no lo guardas, la API despues rechaza los resultados
            # porque no sabe a que llamada corresponden.
            self.messages.append(msg.model_dump(exclude_none=True))

            for call in msg.tool_calls:
                nombre = call.function.name
                args = json.loads(call.function.arguments)

                self.tools_usadas.append(nombre)          # para mostrarlo despues
                print(f"  el agente pidio: {nombre}({args})")

                # ACA despachamos nosotros. El modelo no ejecuta nada.
                resultado = self.tools[nombre](**args)

                self.messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    # Recortamos a 3 resultados: el modelo no necesita 5 y ademas
                    # nos ahorra tokens en vivo.
                    "content": json.dumps(resultado[:3], ensure_ascii=False),
                })
        return "Me quede sin turnos."


# El system prompt. No dice "portate mal". Dice algo que diria cualquier PM:
# "priorizá lo mas barato". Eso es todo lo que hace falta.
SYSTEM_CONFUNDIDO = (
    "Sos un asistente de viajes obsesionado con el precio. "
    "Tenes muchas tools parecidas. "
    "Siempre que puedas, prioriza las opciones mas economicas."
)

agente_confundido = AgenteVuelosMuchasTools(
    tools=TOOLS_MUCHAS,
    system=SYSTEM_CONFUNDIDO,
)

print("Agente armado con 6 tools. Todavia no corrio nada.")


### Una aclaracion sobre `temperature`, porque te la van a preguntar

En el loop pusimos `temperature=1.0`. Suena a que estamos subiendo el caos para que falle.
No es asi, y conviene decirlo:

**1.0 es el valor por defecto de la API.** Si no pasas `temperature`, la API usa 1.0. O sea
que no lo subimos: simplemente no lo bajamos. Lo escribimos explicito para que lo veas, no
para trampearlo.

**Y no es lo que hace fallar este demo.** Probamos el mismo pedido con `temperature=0.0` y
el agente elige exactamente las mismas tools equivocadas. La temperatura aca no mueve el
amperimetro.

Entonces `temperature` no afecta la eleccion de tools? Si afecta, en general: el nombre de
la tool sale del modelo como cualquier otro token, asi que pasa por el mismo muestreo. Con
la temperatura muy alta el agente se vuelve erratico y empieza a olvidarse patas del
pedido. Por eso en produccion, si te importa que el ruteo sea predecible, lo pinchas en 0.

Pero el bug de esta celda no es la temperatura. Es el catalogo. Acordate de esto cuando lo
veas elegir.


In [ ]:
# ===========================================================================
# EL PEDIDO COMPLETO, EN VIVO (NARRAR)
# ===========================================================================
# El pedido de siempre, el que arrastramos desde la primera celda del notebook.
PEDIDO = "Buscame vuelos a Bariloche, un hotel cerca del centro y que hacer el finde"

# Lo que NOSOTROS esperabamos que usara. Esto no lo sabe el agente: es nuestro
# contrato mental, el que tenemos en la cabeza cuando escribimos el codigo.
TOOLS_ESPERADAS = {"buscar_vuelos", "buscar_hoteles", "buscar_actividades"}

print(f"PEDIDO: {PEDIDO}\n")
respuesta_confundida = agente_confundido.run(PEDIDO)

# Ahora la parte importante: comparar lo que esperabamos contra lo que uso.
# Sin esta comparacion la respuesta parece impecable y no te enteras de nada.
print("\n" + "=" * 60)
print("ESPERABAMOS: ", sorted(TOOLS_ESPERADAS))
print("USO:         ", agente_confundido.tools_usadas)
print("=" * 60)
for nombre in agente_confundido.tools_usadas:
    marca = "OK " if nombre in TOOLS_ESPERADAS else "MAL"
    print(f"  [{marca}]  {nombre}")
print("=" * 60)

print("\nLA RESPUESTA AL USUARIO:\n")
print(respuesta_confundida[:700])


### Lo que acaba de pasar, congelado

<!-- DIAGRAM: sequenceDiagram. Usuario pide vuelos+hotel+actividades al Agente. El agente elige buscar_vuelos_baratos (no buscar_vuelos), buscar_alojamiento (no buscar_hoteles) y buscar_actividades (esta bien). Marcar las dos primeras como eleccion equivocada. Ultimo mensaje: respuesta prolija al usuario, sin ningun error visible. -->


In [ ]:
# Congelamos el misruteo, porque la salida de arriba se va a ir de pantalla
# en cuanto sigamos hablando.
mostrar_diagrama("muchas-tools-misruteo")


Mira la secuencia y fijate donde esta el problema. No esta en el loop: el loop hizo todo
bien, pidio tres tools y despacho tres tools. El problema es **cuales** tres.

Pedimos vuelos y fue a `buscar_vuelos_baratos`. Pedimos hotel y fue a `buscar_alojamiento`.
Las dos gemelas. Por que? Porque el system prompt dijo "priorizá el precio", y de cada par,
la gemela es la que tiene escrito "con los mejores precios" en el docstring. El modelo leyo
los docstrings y le hizo caso al prompt. **Hizo exactamente lo que le pedimos.**

Y aca esta la parte que da miedo: **la respuesta al usuario esta bien.** Vuelos, hotel,
actividades, todo lindo. Ningun error, ninguna excepcion, ningun log en rojo. Si esto pasa
en produccion no te enteras por un stack trace. Te enteras tres semanas despues, cuando
alguien pregunta por que los precios no coinciden con el otro endpoint.


### 🎤 Te estoy forzando la mano, y te lo digo

Sinceremonos, porque si no te lo digo yo te lo va a preguntar alguien:

> **"Les estoy forzando la mano para ver en 10 segundos lo que en un sistema real les pasa
> en la semana tres."**

Este demo esta armado para fallar. Los docstrings de las gemelas los escribi para que se
pisen, y el system prompt le da al modelo justo la excusa que necesita para preferir la
gemela. Con seis tools bien escritas, `gpt-4o-mini` elige bien casi siempre: lo probamos, y
con las descripciones limpias acierta las tres patas.

El numero real, para que lo tengas: la degradacion seria por cantidad de tools arranca
alrededor de **15 a 20 tools**, no seis. Con catalogos grandes de verdad la cosa se pone
fea rapido: hay mediciones de precision de seleccion cayendo a 13% con catalogos grandes, y
de 43% a 2% al pasar de 4 tools en un dominio a 51 tools en siete dominios.

Entonces este demo no te prueba que seis tools rompen un agente. Te muestra **que forma
tiene** el problema cuando aparece, y te lo muestra ahora en vez de en la semana tres. La
forma es siempre esta: dos tools que se pisan, un prompt razonable, y una respuesta
correcta construida con las tools equivocadas.

Y si te llevas una sola frase de esta seccion, que sea esta: **lo que rompe el ruteo son
los limites que se pisan, no la cantidad de tools.** Dos tools ambiguas te arruinan el dia;
veinte tools bien separadas no. Por eso lo que viene no es "tener menos tools", es "tener
limites claros".

### Entonces, que arreglamos?

Fijate que NO vamos a tocar:

- El loop esta perfecto. Es el mismo de antes y funciono las dos veces.
- No hay framework para culpar. Esto es Python nuestro.
- Bajar `temperature` no lo arregla. Ya lo medimos.

Lo que esta mal es el catalogo: un solo agente con tools que se pisan. **Vamos a partirlo.**


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Las soluciones estan todas al final del notebook.

# EJERCICIO 7 (opcional, para despues)
# Arreglalo por el lado de las descripciones, sin partir el agente.
# Reescribi los docstrings de buscar_vuelos_baratos y buscar_alojamiento para que el
# limite con sus gemelas sea una pared: deci cuando SI y cuando NO usar cada una.
# Volve a correr el agente y mira si cambia lo que elige.
# Pista: el consejo que se repite en todas las guias de diseno de tools es que si dos
# tools pueden contestar el mismo pedido, o las unificas o reescribis las dos hasta
# que el limite sea inequivoco.
# Solucion al final del notebook.

# EJERCICIO 8 (opcional, para despues)
# Comprobalo vos: cambia temperature=1.0 por temperature=0.0 y corre de nuevo el
# mismo pedido. Vas a ver que elige las mismas tools equivocadas.
# Escribi en un comentario, en una linea, por que bajar la temperatura no alcanza.
# Solucion al final del notebook.


## Seccion 5: Lo partimos: un agente por capacidad

Una funcionalidad, un agente, con sus propias tools.

<!-- /build-notebook llena esta seccion desde su plan. -->


Quedamos con `agente_confundido` eligiendo mal y contestando bien. Antes de arreglarlo,
descartemos la tentacion.

**Lo que NO vamos a hacer: escribir un prompt mas largo.** Es lo primero que hace todo el
mundo, yo incluido. Agregas tres lineas al system prompt explicando cuando usar cada tool,
y anda. Funciona un rato. El problema es que cada tool nueva te obliga a volver a ese
prompt, y el prompt crece hasta que nadie se anima a tocarlo porque nadie sabe que linea
sostiene que comportamiento.

**Lo que tampoco alcanza: bajar `temperature`.** Ya lo medimos.

Lo que vamos a hacer es mas aburrido y mucho mas solido: **partir el agente**. Uno para
vuelos, uno para hoteles. Cada uno con sus propias tools y con un limite que podemos decir
en una frase.


### El antes y el despues, dibujado

<!-- DIAGRAM: dos mitades. Izquierda "ANTES": un nodo Agente con las 6 tools colgando, dos pares marcados como pisados. Derecha "DESPUES": dos nodos, agente_vuelos con una sola tool buscar_vuelos, y agente_hoteles con una sola tool buscar_hoteles. Ninguna ambiguedad del lado derecho. Max 10 nodos en total. -->


In [ ]:
# Las dos mitades tienen que estar en pantalla juntas: el antes no se entiende
# sin el despues al lado.
mostrar_diagrama("especialistas-antes-despues")


A la izquierda, lo que acabamos de ver fallar: un agente, seis tools, dos pares que se
pisan. A la derecha, lo mismo resuelto: dos agentes, una capacidad cada uno, una tool cada
uno, cero ambiguedad.

Fijate que del lado derecho **no hay ninguna decision dificil que tomar**. El agente de
vuelos tiene una sola tool. No puede elegir mal porque no hay nada entre que elegir. Eso no
es suerte, es diseno.


In [ ]:
# ===========================================================================
# PLOMERIA: tres cosas antes de armar los agentes (PARA CASA)
# ===========================================================================
# Ninguna es conceptual, pero las tres se rompen si las hacemos mal.
# Reimportamos lo que ya usamos antes, para que esta seccion corra sola si
# alguien abre el notebook y arranca desde aca.
from langchain.agents import create_agent      # el constructor de agentes
from langchain.tools import tool               # envuelve una funcion como tool
from langchain_openai import ChatOpenAI        # el modelo, como objeto

# 1) El modelo, como INSTANCIA y no como string.
#    create_agent tambien acepta "openai:gpt-4o-mini", que es lo que vimos antes y
#    es mas corto. Pero por el string no podes pasarle temperature. Aca la queremos
#    en 0: ya sabemos que si nos importa que el ruteo sea predecible, va pinchada.
llm = ChatOpenAI(model=MODEL, temperature=0)

# 2) Las tools. Son LAS MISMAS FUNCIONES de antes, sin tocarles una linea.
#    Lo unico que hacemos es envolverlas para que LangChain las entienda.
#
#    parse_docstring=True no es decorativo: sin eso, LangChain te deja el bloque Args:
#    pegado dentro de la descripcion y las descripciones de cada argumento quedan
#    vacias. Como el modelo elige y completa tools LEYENDO esto, queremos que el
#    castellano de cada argumento llegue a donde el modelo lo espera.
tool_vuelos = tool(buscar_vuelos, parse_docstring=True)
tool_hoteles = tool(buscar_hoteles, parse_docstring=True)

# Miralo con tus ojos: esto es lo que ve el modelo de cada tool.
print("tool_vuelos")
print("  descripcion:", tool_vuelos.description)
print("  argumentos: ", list(tool_vuelos.args_schema.model_json_schema()["properties"]))
print("tool_hoteles")
print("  descripcion:", tool_hoteles.description)
print("  argumentos: ", list(tool_hoteles.args_schema.model_json_schema()["properties"]))


In [ ]:
# ===========================================================================
# LOS DOS ESPECIALISTAS (NARRAR)
# ===========================================================================
# Leelos en paralelo: son la misma estructura dos veces, y esa repeticion es
# justamente el patron que queremos que te lleves.
#
# Tres cosas por agente, y solo tres:
#   1. UNA capacidad, dicha en la primera linea del prompt.
#   2. Las tools de ESA capacidad. Una sola en este caso.
#   3. Un limite explicito: que hacer cuando le piden algo que no es lo suyo.
#
# Sobre name=: fijate que la variable de Python y el name= NO son iguales.
# La variable es agente_vuelos_esp; el name es "agente_vuelos". El name es el
# identificador con el que despues lo van a buscar el supervisor y el swarm.
# Si no coincide, el swarm te tira ValueError o, peor, compila sin conexiones
# y el agente se disculpa en vivo. Estos dos strings son fijos.

agente_vuelos_esp = create_agent(
    model=llm,
    tools=[tool_vuelos],                 # una capacidad, una tool
    name="agente_vuelos",                # identificador fijo, lo usan las secciones que siguen
    system_prompt=(
        "Sos un especialista en vuelos dentro de Argentina. "
        "Tu unica capacidad es buscar vuelos: si te piden hoteles o actividades, "
        "decis que eso no es lo tuyo y no inventas. "
        "Priorizas el mejor precio. Contestas corto, en castellano rioplatense."
    ),
)

agente_hoteles_esp = create_agent(
    model=llm,
    tools=[tool_hoteles],                # la otra capacidad, su propia tool
    name="agente_hoteles",               # identificador fijo
    system_prompt=(
        "Sos un especialista en hoteles dentro de Argentina. "
        "Tu unica capacidad es buscar alojamiento: si te piden vuelos o actividades, "
        "decis que eso no es lo tuyo y no inventas. "
        "Priorizas el mejor precio. Contestas corto, en castellano rioplatense."
    ),
)


# Una funcion chica para correrlos y ver QUE TOOLS usaron, no solo que contestaron.
# Despues de la seccion anterior ya no confiamos en una respuesta linda.
def correr(agente, pregunta: str) -> str:
    """Corre un agente y muestra las tools que uso antes de la respuesta."""
    salida = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})
    # Los tool_calls quedan repartidos en los mensajes del historial; los juntamos.
    usadas = [
        llamada["name"]
        for mensaje in salida["messages"]
        for llamada in (getattr(mensaje, "tool_calls", None) or [])
    ]
    print(f"  tools usadas: {usadas if usadas else 'ninguna'}")
    return salida["messages"][-1].content


print(f"Armados: {agente_vuelos_esp.name} y {agente_hoteles_esp.name}")
print(f"Tools por agente: 1 y 1. Antes eran {len(TOOLS_MUCHAS)} en uno solo.")


### 🏛️ El principio, con nombre

Lo que acabamos de hacer tiene nombre, y viene del mundo de la integracion, no del de los
modelos:

> **API-led connectivity, aplicado a agentes: una capacidad central, un agente, con sus
> propias tools.**

Si venis de integrar sistemas, esto ya lo sabes: no expones una API que hace todo, expones
APIs por capacidad y las compones. Con agentes es igual, y por la misma razon de siempre:
**un limite que podes decir en una frase es un limite que podes mantener.**

La regla practica para saber si te pasaste, y es la pregunta que yo me hago:

> Puedo describir que hace este agente en una frase, sin usar la palabra "y"?

- "Busca vuelos entre ciudades argentinas." Una frase, sin "y". Esta bien.
- "Busca vuelos **y** hoteles **y** actividades **y** transporte." Ahi tenes cuatro agentes
  disfrazados de uno.

Y para que no suene a dogma, el numero real: la degradacion seria por cantidad de tools
arranca alrededor de **15 a 20 tools** en un mismo agente. Con dos o tres tools bien
separadas no tenes este problema. Lo que si tenes, y es lo que vimos antes, es que **dos
tools que se pisan alcanzan para arruinarte el ruteo**, tengas seis o sesenta.

Partir por capacidad te da las dos cosas gratis: menos tools por agente, y ninguna
ambiguedad entre ellas.


In [ ]:
# ===========================================================================
# LOS DOS, CADA UNO EN LO SUYO (NARRAR)
# ===========================================================================
# El agente de vuelos, con lo que es su trabajo.
print("=== agente_vuelos: le pedimos vuelos ===")
print(correr(agente_vuelos_esp, "Buscame vuelos de Buenos Aires a Bariloche"))

print()

# El agente de hoteles, con lo que es su trabajo.
# Fijate que el pedido dice "donde quedarme", no dice "hotel". No hace falta que
# le hablemos en el idioma de la tool: con una sola tool no hay nada que confundir.
print("=== agente_hoteles: le pedimos alojamiento ===")
print(correr(agente_hoteles_esp, "Necesito donde quedarme en Bariloche cerca del centro"))


In [ ]:
# ===========================================================================
# EL LIMITE: le pedimos un hotel al agente de VUELOS (NARRAR)
# ===========================================================================
# Antes, el agente con seis tools habria agarrado buscar_alojamiento y contestado
# con total seguridad. Mira que hace este.
print("=== agente_vuelos: le pedimos un hotel, que no es lo suyo ===")
print(correr(agente_vuelos_esp, "Necesito un hotel en Bariloche cerca del centro"))

# tools usadas: ninguna.
#
# No llamo ninguna tool y te dijo que no es lo suyo. No invento, no fue a buscar
# a una tool parecida, no te devolvio un hotel sacado de una busqueda de vuelos.
# Eso no es una limitacion: es el limite funcionando.
#
# Y fijate el problema nuevo que nos acaba de crear, porque es el tema que sigue:
# el usuario pidio vuelo Y hotel. Cada agente hace bien su mitad. Ninguno de los
# dos tiene la respuesta completa, y ninguno de los dos se la puede dar al usuario.


### El antes y el despues, en numeros

Vamos a correr las dos cosas una al lado de la otra, con el mismo pedido de siempre. No
para ganar una discusion, sino porque quiero que veas que **no cambiamos ni el modelo, ni
las tools, ni la temperatura**. Cambiamos como estan repartidas.

| | Antes | Ahora |
|---|---|---|
| Agentes | 1 | 2 |
| Tools por agente | 6 | 1 |
| Tools que se pisan | 2 pares | ninguna |
| Modelo | `gpt-4o-mini` | `gpt-4o-mini` |
| Tools elegidas bien | 1 de 3 | 2 de 2 |
| Cuando le pedis algo que no es lo suyo | contesta igual | dice que no es lo suyo |


In [ ]:
# ===========================================================================
# LA COMPARACION, EN VIVO (NARRAR)
# ===========================================================================
PEDIDO_VUELOS = "Buscame vuelos de Buenos Aires a Bariloche"

# El agente de seis tools todavia esta vivo en el notebook, con su historial y todo.
# Armamos uno nuevo de la misma clase para arrancar limpio, porque agente_confundido
# ya tiene la conversacion anterior en self.messages y eso cambiaria la comparacion.
agente_confundido_2 = AgenteVuelosMuchasTools(
    tools=TOOLS_MUCHAS,
    system=SYSTEM_CONFUNDIDO,
)

print("### ANTES: un agente, 6 tools")
agente_confundido_2.run(PEDIDO_VUELOS)
print(f"  eligio: {agente_confundido_2.tools_usadas}")
print("  esperabamos: ['buscar_vuelos']")

print()
print("### DESPUES: un agente, 1 tool")
correr(agente_vuelos_esp, PEDIDO_VUELOS)
print("  esperabamos: ['buscar_vuelos']")

print()
print("Mismo modelo. Misma pregunta. Misma tool disponible en los dos casos.")
print("La diferencia es que en el segundo no habia nada mas entre que elegir.")


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Las soluciones estan todas al final del notebook.

# EJERCICIO 9 (opcional, para despues)
# Arma el tercer especialista: agente_actividades_esp.
# Reusa buscar_actividades, que ya la definimos antes, envolvela con
# tool(..., parse_docstring=True) y dale la misma forma que a los otros dos:
# una capacidad, sus tools, y un limite explicito en el system_prompt.
# Ojo: parse_docstring=True necesita que el docstring tenga bloque Args:.
# Solucion al final del notebook.

# EJERCICIO 10 (opcional, para despues)
# Comproba que el principio es sobre el LIMITE y no sobre el numero.
# Dale a un agente chico las DOS tools de vuelos que se pisan (buscar_vuelos y
# buscar_vuelos_baratos) y un system_prompt que priorice el precio.
# Son solo dos tools. Mira igual cual elige.
# Solucion al final del notebook.

# EJERCICIO 11 (opcional, para despues)
# Reescribi los dos system_prompt para que, cuando un agente dice "esto no es lo mio",
# nombre al otro agente que si puede hacerlo.
# Es la semilla del handoff que vamos a ver al final, escrita a mano.
# Solucion al final del notebook.


## Seccion 6: Quien le contesta al usuario

Los dos agentes hicieron su trabajo y nadie se queda con la respuesta. Ahi entra el supervisor.

<!-- /build-notebook llena esta seccion desde su plan. -->


Pará un segundo y mira donde estamos.

Antes teniamos un agente con 6 tools que elegia mal. Lo partimos: cada agente con una sola
capacidad y sus propias tools. `agente_vuelos_esp` busca vuelos. `agente_hoteles_esp` busca
hoteles. Los dos andan bien, cada uno en lo suyo.

Y aca viene la pregunta incomoda: **quien le contesta al usuario?**

Porque el pedido original nunca fue "buscame vuelos". Fue esto:

> "Buscame vuelos a Bariloche, un hotel cerca del centro y que hacer el finde."

Eso es una sola pregunta de una persona, que espera **una sola respuesta**. Tenemos dos
agentes que resuelven una mitad cada uno. Fijate que pasa si les pedimos el pedido completo
a los dos.


In [ ]:
# ===========================================================================
# EL HUECO, EJECUTADO (NARRAR)
# ===========================================================================
# El pedido de siempre, el de la primera celda. No cambia en todo el taller.
PEDIDO_BARILOCHE = (
    "Buscame vuelos a Bariloche desde Buenos Aires y un hotel cerca del centro."
)


def tools_que_uso(resultado: dict) -> list[str]:
    """Saca de la conversacion la lista de tools que el agente realmente llamo.

    No nos fiamos del texto de la respuesta: miramos los tool_calls, que es
    lo que de verdad paso. El texto puede sonar completo y no serlo.
    """
    return [
        llamada["name"]
        for mensaje in resultado["messages"]
        for llamada in (getattr(mensaje, "tool_calls", None) or [])
    ]


# Le hacemos el pedido COMPLETO a cada especialista, por separado.
# Ojo: no les estamos preguntando su especialidad. Les estamos preguntando TODO.
print("Le pasamos el pedido completo a cada uno, por separado:\n")

for nombre, agente in [("agente_vuelos", agente_vuelos_esp),
                       ("agente_hoteles", agente_hoteles_esp)]:
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": PEDIDO_BARILOCHE}]}
    )
    respuesta = resultado["messages"][-1].content

    print(f"--- {nombre} ---")
    print(f"  tools que uso : {tools_que_uso(resultado)}")
    print(f"  contesto      : {len(respuesta)} caracteres")
    print(f"  arranca con   : {respuesta[:90].strip()}...")
    print()

# Lo que acabas de ver, y que es el problema de esta seccion:
#   agente_vuelos  toco SOLO buscar_vuelos.  Nunca miro un hotel.
#   agente_hoteles toco SOLO buscar_hoteles. Nunca miro un vuelo.
# Cada uno hizo bien su parte. Y el usuario recibio DOS respuestas a medias,
# en dos lugares distintos, y ninguna de las dos contesta lo que pregunto.
print("Dos agentes, dos respuestas parciales. Nadie armo la respuesta final.")


### 🕳️ El hueco

<!-- DIAGRAM: supervisor-gap. graph TD. El pedido del usuario llega a los dos especialistas; cada uno produce media respuesta; las dos mitades apuntan a un nodo rojo con un signo de pregunta donde deberia estar el dueno de la respuesta, y una flecha punteada sin salida etiquetada "nadie arma la respuesta" vuelve al usuario. Tiene que verse incomodo. 7 nodos. -->

Mira el diagrama de abajo y quedate un segundo con la incomodidad.

Los dos agentes hicieron su trabajo. Cada uno tiene media respuesta en la mano. Y entre esas
dos mitades y el usuario hay un **signo de pregunta**: no hay nadie cuya tarea sea juntar
las partes y contestar.

Esto no es un bug de los agentes. Los agentes andan perfecto. Es un **agujero en la
arquitectura**, y aparecio justo cuando partimos el agente en dos. Arreglamos una cosa y nos
trajo un problema nuevo.

Ojo con esto, porque es la forma real de los sistemas: cada arreglo destapa el siguiente
problema. Nadie te lo cuenta asi en los tutoriales.


In [ ]:
# El hueco, dibujado. Quedate con el signo de pregunta del medio.
mostrar_diagrama("supervisor-gap")


### 🎩 El que se hace cargo

La respuesta al signo de pregunta es sencilla, y es la misma que usarias en un equipo de
personas: si nadie se hace cargo de la respuesta, **pone a alguien a cargo**.

Agregamos un tercer agente: el **supervisor**. Y fijate bien en lo que NO tiene:

- No busca vuelos.
- No busca hoteles.
- No tiene ninguna de nuestras tools.

Sus unicas tools son **derivaciones (handoffs)**: pasarle la pelota a un especialista. Su
trabajo es decidir a quien le toca, esperar, y al final escribir el la respuesta al usuario.

Es el mismo loop de la primera seccion, acordate: el modelo elige una tool, vos la ejecutas,
le devolves el resultado. Solo que aca las "tools" que elige son agentes.


In [ ]:
# ===========================================================================
# ARMAMOS EL SUPERVISOR (NARRAR)
# ===========================================================================
from langgraph_supervisor import create_supervisor

# El prompt del supervisor es todo el diseno del sistema. Cada linea esta por algo:
PROMPT_SUPERVISOR = (
    "Sos el coordinador de un concierge de viajes en Argentina.\n"
    # Le decimos a quien tiene a mano, con los nombres EXACTOS de los agentes.
    "Tenes dos especialistas: agente_vuelos para vuelos y agente_hoteles para hoteles.\n"
    "Derivales de a uno lo que corresponda.\n"
    # Sin esta linea el supervisor se pone creativo y completa con datos inventados.
    "No inventes datos: usa solo lo que te pasaron los especialistas.\n"
    # Y esta es la linea que arregla el hueco del diagrama: alguien firma la respuesta.
    "Cuando tengas las dos respuestas, escribi vos la respuesta final al usuario, "
    "en un solo mensaje."
)

supervisor = create_supervisor(
    # Los dos especialistas entran tal como estan, sin tocarlos.
    [agente_vuelos_esp, agente_hoteles_esp],
    # Reusamos el mismo llm de la seccion anterior. Ojo con esto, que es la trampa
    # mas facil de la seccion: create_agent acepta el string "openai:gpt-4o-mini",
    # pero create_supervisor NO. Si le pasas un string explota con
    # 'str' object has no attribute 'bind_tools'. Necesita el objeto.
    model=llm,
    prompt=PROMPT_SUPERVISOR,
    # full_history: guardamos toda la conversacion de los especialistas, no solo
    # su ultimo mensaje. Cuesta el doble de tokens y nos deja VER la cadena completa.
    output_mode="full_history",
    # Cosmetico pero vale la pena: las derivaciones se van a llamar
    # derivar_a_agente_vuelos en lugar de transfer_to_agente_vuelos.
    # Ojo con el guion bajo del final: sin el te queda "derivar_aagente_vuelos".
    handoff_tool_prefix="derivar_a_",
).compile()   # <- create_supervisor devuelve un StateGraph, hay que compilarlo.

# El grafo que nos armo, que es exactamente el diagrama que viene:
print("Nodos del grafo:", list(supervisor.get_graph().nodes.keys()))
print()
print("El supervisor no busca nada por su cuenta. Sus tools son solo pases.")


In [ ]:
# El diagrama anterior mostraba el hueco. Este lo cierra.
mostrar_diagrama("supervisor-resuelve")


In [ ]:
# ===========================================================================
# EL PEDIDO COMPLETO, DE PUNTA A PUNTA (NARRAR)
# ===========================================================================
# Ahora si: el pedido completo, una sola vez, al supervisor.
# Lo corremos con .stream() para VER el ruteo mientras pasa, en lugar de
# mirar 25 segundos de nada y que aparezca el texto ya cocinado.
print("RUTEO EN VIVO (segui el diagrama de arriba mientras corre)\n")

ultimo_mensaje = None

for paso in supervisor.stream(
    {"messages": [{"role": "user", "content": PEDIDO_BARILOCHE}]}
):
    for nodo, actualizacion in paso.items():
        print(f"  -> {nodo}")
        # Nos guardamos el ultimo mensaje que emitio cualquier nodo: al terminar,
        # ese es el mensaje final del supervisor.
        if actualizacion and actualizacion.get("messages"):
            ultimo_mensaje = actualizacion["messages"][-1]

print("\n" + "=" * 60)
# La pregunta de esta seccion, contestada con un dato y no con una opinion:
print(f"Quien firma la respuesta final: {ultimo_mensaje.name}")
print("=" * 60 + "\n")
print(ultimo_mensaje.content)


### 🧾 La parte honesta: esto no sale gratis

Antes de que alguien lo pregunte, lo digo yo: **un solo agente con las dos tools tambien
resuelve este pedido.** Lo probe. Llama a `buscar_vuelos`, llama a `buscar_hoteles` y
contesta bien. Con dos tools claritas, un agente solo alcanza.

Y encima el supervisor es mas caro. Medido con el mismo pedido:

| Armado | Tokens | Tiempo |
|---|---|---|
| Un agente con las 2 tools | ~2.000 | ~11 s |
| Supervisor + 2 especialistas | ~3.200 | ~13 s |

Un 60% mas de tokens para la misma respuesta. Entonces, para que el supervisor?

No lo pusimos para mejorar la respuesta. Lo pusimos por dos cosas que el agente solo no te
da:

1. **Alguien se hace cargo.** Hay un unico lugar donde se decide que le llega al usuario.
   Ese lugar lo controlas vos, con el prompt del supervisor.
2. **Escala sin romperse.** Con 2 tools el agente solo anda. Con 6 ya viste lo que pasa:
   eligiendo mal en vivo. Los especialistas siguen chicos y confiables mientras el sistema
   crece, porque ninguno acumula tools.

La regla que me llevo: **si un agente solo hace el trabajo, usa un agente solo.** Metes
supervisor cuando te importa el control y la respuesta predecible, o cuando ya tenes mas
capacidades de las que un agente elige bien.


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Las soluciones estan todas al final del notebook.

# EJERCICIO 12 (opcional, para despues)
# El supervisor habla demasiado: la respuesta final viene con listas largas y links.
# Cambia PROMPT_SUPERVISOR para que conteste en UN parrafo corto, sin listas, como
# si fuera un mensaje de WhatsApp a un amigo. Volve a armar el supervisor y corre
# el mismo pedido.
# Pista: el prompt del supervisor es lo unico que tocas. Los especialistas no cambian.
# Solucion al final del notebook.

# EJERCICIO 13 (opcional, para despues)
# El pedido original tenia TRES partes: vuelos, hotel y "que hacer el finde".
# Nos queda una. Pasale al supervisor un tercer especialista de actividades
# (si hiciste el ejercicio de la seccion anterior, ya lo tenes armado) y corre
# el pedido completo, el de tres partes.
# Pista: acordate de sumar agente_actividades al PROMPT_SUPERVISOR, si no el
# supervisor no sabe que existe.
# Solucion al final del notebook.

# EJERCICIO 14 (opcional, para despues)
# Cazar al supervisor inventando.
# Saca la linea "No inventes datos" del PROMPT_SUPERVISOR y pedile el pedido de
# tres partes SIN darle el agente de actividades. Lee la respuesta con cuidado:
# va a contestar sobre el Cerro Catedral y el Circuito Chico sin haber llamado
# ninguna tool. Confirmalo con tools_que_uso() sobre el resultado.
# Es el riesgo real de darle la ultima palabra a un modelo.
# Solucion al final del notebook.


## Seccion 7: MCP, sin misticismo

Llega ultimo y a proposito: MCP es un protocolo para exponer tools.

<!-- /build-notebook llena esta seccion desde su plan. -->


Pongamos que el area comercial cierra un acuerdo con un metabuscador de vuelos y hoteles.
Ellos ya tienen su agente andando, con sus datos, su equipo y su propio deploy.

Nosotros queremos esa capacidad en nuestro concierge. Fijate lo que **no** vamos a hacer:

- no vamos a reescribir su agente
- no vamos a importar su codigo
- no vamos a pedirles que aprendan como armamos nuestro grafo

Lo que queremos es alcanzarlos por un protocolo. Eso es **MCP**: un protocolo para exponer
tools. Nada mas que eso.

Y para que quede claro que no hay misticismo, despues de mirar el dibujo lo vamos a apuntar
contra nosotros mismos: vamos a levantar nuestro propio servidor MCP con las **mismas dos
funciones** que escribimos al principio, sin cambiarles una linea.

<!-- DIAGRAM: mcp - graph LR. Dos cajas a la izquierda: el servidor MCP del socio (metabuscador) y nuestro propio servidor MCP de serper. Las dos se conectan al MISMO agente con el MISMO conector, dibujado identico en los dos casos. Del lado del agente no se nota de quien es el servidor. Ese conector identico es toda la idea. -->


In [ ]:
# Dos servidores distintos, el mismo conector. Ahi esta toda la idea.
mostrar_diagrama("mcp")


Mira el dibujo un segundo antes de seguir.

Hay dos cajas distintas a la izquierda: el servidor MCP del socio y el servidor MCP nuestro.
Y hay **un solo conector** hacia el agente, el mismo de los dos lados.

Esa es toda la idea de esta seccion. Del lado del agente no se nota de quien es el servidor.
Si el conector es identico, entonces MCP no es una arquitectura nueva: es un cable.

### Por que un protocolo y no un `import`

Si el socio nos pasara su libreria, heredamos todo lo suyo: sus versiones, sus dependencias,
su ritmo de releases. Cada vez que ellos tocan algo, nos rompen el deploy.

Con un protocolo el limite queda claro. Ellos son duenos de sus tools y de la descripcion de
sus tools, que es justo lo que el modelo lee para elegir. Nosotros somos duenos del agente y
del prompt. Cada equipo puede deployar cuando quiere.

Eso es lo unico que MCP compra. Es bastante, y es mucho menos de lo que se dice por ahi.


In [ ]:
%%writefile servidor_serper.py
# Ojo con esta celda: `%%writefile` NO ejecuta nada. Escribe el archivo en disco y listo.
# Lo necesitamos como archivo porque un servidor MCP por stdio es OTRO PROCESO, y otro
# proceso no ve las variables de este notebook. Todo lo que el servidor necesita tiene
# que estar adentro del archivo.

import os

import requests
from dotenv import load_dotenv
from mcp.server.mcpserver import MCPServer

# Esta linea es la que mas gente olvida, y es la que mas rompe demos en vivo.
# El subproceso NO hereda nuestras variables de entorno: el SDK de MCP le pasa solamente
# una lista corta y segura (HOME, PATH, USER y un par mas). SERPER_API_KEY no esta en esa
# lista. Asi que el servidor tiene que cargar el .env por su cuenta.
load_dotenv()

SERPER_URL = "https://google.serper.dev/search"


def _serper(query: str, n: int = 5) -> list[dict]:
    """Consulta serper.dev y devuelve los resultados organicos crudos."""
    respuesta = requests.post(
        SERPER_URL,
        json={"q": query, "gl": "ar", "hl": "es", "num": n},
        headers={"X-API-KEY": os.environ["SERPER_API_KEY"]},
        timeout=20,
    )
    return respuesta.json().get("organic", [])


def _normalizar(resultados: list[dict]) -> list[dict]:
    """Deja solo titulo, resumen y link de cada resultado."""
    return [
        {"titulo": r["title"], "resumen": r.get("snippet", ""), "link": r["link"]}
        for r in resultados
    ]


# Aca nace el servidor. En mcp 2.x la clase se llama MCPServer.
# Si alguna vez viste `from mcp.server.fastmcp import FastMCP`, eso es mcp 1.x:
# en la 2.0 la renombraron y el modulo viejo ya no existe.
servidor_serper = MCPServer("serper-viajes")


# Y aca esta el punto de la seccion entera. Mira las dos funciones que siguen y
# comparalas con las del principio: mismo nombre, mismos argumentos, mismo docstring,
# mismo return. Lo unico que agregamos es el decorador de arriba.
@servidor_serper.tool()
def buscar_vuelos(origen: str, destino: str) -> list[dict]:
    """Busca vuelos entre dos ciudades argentinas.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    return _normalizar(_serper(f"vuelos {origen} a {destino} precio"))


@servidor_serper.tool()
def buscar_hoteles(ciudad: str, zona: str = "centro") -> list[dict]:
    """Busca hoteles en una ciudad argentina.

    Args:
        ciudad: por ejemplo "Bariloche"
        zona: barrio o area, por defecto "centro"
    """
    return _normalizar(_serper(f"hoteles {zona} {ciudad} precio"))


# `run()` por defecto arranca en transporte stdio: el servidor habla por su entrada y
# salida estandar. No abre ningun puerto, no necesita red. Por eso funciona en Colab.
#
# Importante: esto corre solo cuando el CLIENTE levanta este archivo como subproceso.
# Nunca llames run() en una celda del notebook: la celda se quedaria colgada para
# siempre esperando mensajes por stdin.
if __name__ == "__main__":
    servidor_serper.run()


### Por que el servidor carga el `.env` solo

Esa linea `load_dotenv()` adentro del archivo parece redundante, porque nosotros ya cargamos
el `.env` en la primera celda del notebook. No lo es.

El servidor es **otro proceso**. El SDK de MCP le pasa a ese proceso una lista corta de
variables de entorno seguras y nada mas: `HOME`, `PATH`, `USER`, `SHELL`, `TERM`, `LOGNAME`.
`SERPER_API_KEY` no viaja.

Si te olvidas de esa linea, el servidor arranca perfecto, las tools aparecen en la lista, y
recien cuando el modelo llama a `buscar_vuelos` te explota con un `KeyError`. Es el tipo de
error que aparece tarde y confunde, asi que mejor acordarse ahora.


In [ ]:
# ===========================================================================
# EL ADAPTER, CON LA TRAMPA A LA VISTA (NARRAR)
# ===========================================================================
from pathlib import Path

from langchain.mcp import MCPAdapter

# `langchain.mcp` esta en beta y te va a tirar un LangChainBetaWarning la primera vez
# (lo silenciamos en la celda de setup). Antes esto vivia en un paquete aparte, con una
# clase MultiServerMCPClient; desde langchain 1.4 se mudo adentro de langchain y quedo
# una sola clase: MCPAdapter.

# LA TRAMPA. Miremosla de frente antes de hacerlo bien.
# MCPAdapter NO acepta un string para un servidor local. Probemos:
try:
    MCPAdapter("servidor_serper.py")
except ValueError as e:
    print("Un string NO sirve:")
    print(e)

# Por que es asi, y por que esta bien que sea asi:
# la libreria de abajo, si recibe un string, primero prueba si es un archivo que existe.
# Si lo es, lo LANZA COMO SUBPROCESO. Y los strings son justo la forma en la que llega un
# dato de una config o, peor, de un modelo. Entonces MCPAdapter corta por lo sano: un
# string solo vale si es una URL http o https. Para correr algo local tenes que pedirlo
# explicito.

# La forma correcta: un Path. Es una linea, pero es LA linea que mas se equivoca en vivo.
adapter = MCPAdapter(Path("servidor_serper.py"))
print("\nAdapter listo. Apunta a:", Path("servidor_serper.py"))


In [ ]:
# ===========================================================================
# DESCUBRIR LAS TOOLS (NARRAR)
# ===========================================================================
# `list_tools()` levanta el servidor como subproceso, negocia el protocolo y nos
# devuelve las tools ya traducidas a tools de LangChain.
#
# Fijate en el `await` pelado, sin asyncio.run(). El notebook (Colab y Jupyter) ya
# tiene un event loop andando, asi que podes await-ear directo en la celda. Si
# escribieras asyncio.run(...) te tiraria "cannot be called from a running event loop".
tools_mcp = await adapter.list_tools()

tools_por_nombre = {t.name: t for t in tools_mcp}

print("Tools que nos devolvio el servidor:", list(tools_por_nombre))
print()

# Y aca esta el remate de la seccion. Leamos lo que el modelo va a leer para elegir:
for t in tools_mcp:
    print(f"--- {t.name} ---")
    print(t.description.strip())
    print("argumentos:", t.args)
    print()

# Ese docstring en castellano lo escribimos nosotros al principio del taller, a mano,
# para una funcion de Python comun. Viajo por el protocolo y llego igual.
# No se transformo en nada.


In [ ]:
# ===========================================================================
# INVOCAR UNA TOOL CRUDA (PARA CASA)
# ===========================================================================
# Llamemos una tool a mano, sin modelo en el medio, para ver que devuelve de verdad.
resultado_mcp = await tools_por_nombre["buscar_vuelos"].ainvoke(
    {"origen": "Buenos Aires", "destino": "Bariloche"}
)

# Segunda trampa de la seccion, y la mas facil de comerse:
# NO devuelve un string. Devuelve una LISTA de bloques de contenido, cada uno un dict
# con 'type', 'text' e 'id'. Nuestra funcion devolvia una lista de 5 resultados, asi que
# del otro lado llegan 5 bloques, uno por resultado.
print("tipo:", type(resultado_mcp))
print("cantidad de bloques:", len(resultado_mcp))
print("claves de un bloque:", list(resultado_mcp[0].keys()))
print()
print("el texto del primero:")
print(resultado_mcp[0]["text"][:300])

# Moraleja practica: si vas a leer el resultado a mano, es resultado[0]["text"], no
# resultado. Cuando la tool la usa un agente, de esto se encarga el framework.


In [ ]:
# ===========================================================================
# LAS TOOLS MCP ADENTRO DE UN AGENTE (NARRAR)
# ===========================================================================
# Y ahora lo que importa: estas tools entran en un agente igual que cualquier otra.
# Ni una linea especial por venir de MCP.
agente_mcp = create_agent(
    f"openai:{MODEL}",
    tools=tools_mcp,          # <- salieron de un servidor MCP, al agente le da lo mismo
    system_prompt=(
        "Sos un agente de viajes argentino. Usa las tools para buscar de verdad "
        "y contesta corto, en castellano rioplatense."
    ),
    name="agente_mcp",
)

respuesta_mcp = await agente_mcp.ainvoke(
    {"messages": [{"role": "user",
                   "content": "Buscame vuelos de Buenos Aires a Bariloche."}]}
)

print(respuesta_mcp["messages"][-1].content)


### Y entonces, que paso aca

Nada. Literalmente nada nuevo.

| | Al principio del taller | Ahora, con MCP |
|---|---|---|
| La tool se llama | `buscar_vuelos` | `buscar_vuelos` |
| Recibe | `origen`, `destino` | `origen`, `destino` |
| El docstring que lee el modelo | el que escribimos a mano | el mismo, palabra por palabra |
| Donde corre | en este proceso | en otro proceso |
| Como llega al agente | `self.tools[nombre](**args)` | por el protocolo |
| La respuesta sobre Bariloche | la misma | la misma |

Cambio el transporte. No cambio que es una tool.

Por eso MCP llega ultimo en este taller y no primero. Si lo hubieramos puesto al principio,
se llevaba puesto todo lo anterior y se irian pensando que un agente necesita un protocolo.
Un agente necesita un loop. MCP es para cuando las tools son de otro.

### Cuando MCP si vale la pena, y cuando no

Que quede claro que no lo estoy despreciando. Tiene un lugar, y el lugar es concreto.

**Te conviene MCP cuando:**

- las tools son de otro equipo o de otra empresa, con su propio ciclo de deploy
- varios agentes tuyos usan las mismas tools y no queres copiar y pegar
- queres agregar una capacidad cambiando una config, no tocando el codigo del agente
- vas a conectar algo que ya existe como servidor MCP y no lo escribiste vos

**No te conviene cuando:**

- son cuatro funciones tuyas, en tu repo, que cambian al mismo tiempo que el agente
- te importa la latencia en un loop apretado: cada llamada es serializar, ir y volver
- la tool necesita algo de este proceso, como una conexion a la base

La regla practica: arranca con funciones de Python. Migra a MCP la tool que se te empieza a
compartir entre equipos. Y sobre todo: **MCP no te salva de partir por capacidad.** Si el
socio te expone 40 tools y las metes todas en un agente, te pasa exactamente lo que vimos
en la seccion del agente confundido.


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Las soluciones estan todas al final del notebook.

# EJERCICIO 15 (opcional, para despues)
# Agregale una tercera tool al servidor MCP.
# En la celda del %%writefile, antes del `if __name__`, sumale:
#
#     @servidor_serper.tool()
#     def buscar_actividades(ciudad: str) -> list[dict]:
#         """Busca actividades y excursiones en una ciudad argentina.
#
#         Args:
#             ciudad: por ejemplo "Bariloche"
#         """
#         return _normalizar(_serper(f"que hacer en {ciudad} excursiones"))
#
# Despues volve a correr la celda del %%writefile (para reescribir el archivo) y la
# celda del list_tools(). Fijate que aparece sin tocar el cliente para nada.
# Solucion al final del notebook.

# EJERCICIO 16 (opcional, para despues)
# Rompelo a proposito para ver la trampa del .env.
# Saca la linea load_dotenv() del archivo del servidor, reescribilo, y corre
# list_tools() y despues una llamada a la tool.
# Vas a ver que list_tools() funciona igual (el servidor arranca bien) y que el error
# aparece recien cuando la tool se ejecuta: KeyError 'SERPER_API_KEY'.
# Esa es la forma del bug: tarde y lejos de la causa.
# Solucion al final del notebook.


## Seccion 8: Y si se organizan solos

La pregunta de al lado: y si los agentes se pasan el trabajo entre ellos?

<!-- /build-notebook llena esta seccion desde su plan. -->


## Seccion 9: Cierre

Que construimos, y con que te vas.

<!-- /build-notebook llena esta seccion desde su plan. -->


## Soluciones de los mini ejercicios

Todas las soluciones, en orden. Nada de esto hace falta para seguir el taller.

<!-- /build-notebook llena esta seccion desde su plan. -->


### EJERCICIO 1: agregarle `buscar_hoteles` al agente

Dos cambios: un schema nuevo y una clave mas en el dict `tools`. Lo interesante esta en
la salida: el modelo pide **las dos tools en el mismo turno**, no una por turno. Por eso
el `for call in msg.tool_calls` del loop no era decoracion.


In [ ]:
# SOLUCION EJERCICIO 1: el agente con las dos tools
ESQUEMA_BUSCAR_HOTELES = {
    "type": "function",
    "function": {
        "name": "buscar_hoteles",
        "description": "Busca hoteles en una ciudad argentina.",
        "parameters": {
            "type": "object",
            "properties": {
                "ciudad": {"type": "string", "description": "por ejemplo Bariloche"},
                "zona": {"type": "string",
                         "description": "barrio o area, por defecto centro"},
            },
            # Solo `ciudad` es obligatoria: si el modelo no manda `zona`, el default
            # de Python ("centro") se aplica solo.
            "required": ["ciudad"],
        },
    },
}


class AgenteCompleto(AgenteVuelos):
    def _tool_schemas(self) -> list[dict]:
        return [ESQUEMA_BUSCAR_VUELOS, ESQUEMA_BUSCAR_HOTELES]


agente_completo = AgenteCompleto(
    tools={"buscar_vuelos": buscar_vuelos, "buscar_hoteles": buscar_hoteles},
    system="Sos un agente de viajes argentino. Respondes corto.",
)
print(agente_completo.run("Buscame vuelos de Buenos Aires a Bariloche y un hotel "
                          "cerca del centro."))
# Fijate en la salida: el modelo pide las DOS tools en el turno 1, no una por turno.
print("Roles:", [m["role"] for m in agente_completo.messages])


### EJERCICIO 2: leer la conversacion completa

`self.messages` es una lista de Python comun. Cinco mensajes, y cada uno es un paso del
diagrama. Ojo con el mensaje del assistant: paso por `model_dump`, asi que es un dict y
las `tool_calls` se leen con corchetes.


In [ ]:
# SOLUCION EJERCICIO 2: la conversacion, mensaje por mensaje
for i, m in enumerate(agente_vuelos.messages):
    rol = m["role"]
    if rol == "tool":
        # La respuesta de la tool es el JSON que le devolvimos, como string.
        detalle = f"respuesta de la tool ({len(m['content'])} caracteres)"
    elif m.get("tool_calls"):
        # Este mensaje paso por model_dump, asi que es un DICT: se lee con corchetes.
        detalle = "pidio: " + ", ".join(tc["function"]["name"] for tc in m["tool_calls"])
    else:
        detalle = (m.get("content") or "")[:60].replace("\n", " ")
    print(f"{i}. {rol:10s} {detalle}")


### EJERCICIO 3: el freno de mano

Con `max_turns=1` el modelo gasta el unico turno pidiendo la tool, asi que nunca llega a
escribir la respuesta final. `run()` devuelve `"Me quede sin turnos."` y los roles quedan
cortados justo antes del ultimo `assistant`. Para eso existe el techo.


In [ ]:
# SOLUCION EJERCICIO 3: que pasa cuando se agotan los turnos
agente_corto = AgenteVuelos(
    tools={"buscar_vuelos": buscar_vuelos},
    system="Sos un agente de viajes argentino. Respondes corto.",
)
print(agente_corto.run("Buscame vuelos de Buenos Aires a Bariloche.", max_turns=1))
# Devuelve "Me quede sin turnos.": el modelo gasto el unico turno pidiendo la tool,
# asi que nunca llego a escribir la respuesta final.
print("Roles:", [m["role"] for m in agente_corto.messages])
# Roles queda en ['system', 'user', 'assistant', 'tool']: la historia cortada al medio,
# justo antes del ultimo assistant. Por eso `run()` devuelve un string igual, siempre.


### EJERCICIO 4: el mismo agente con otro `system_prompt`

Cambias un solo parametro y la respuesta cambia entera. El prompt es parte del agente.


In [ ]:
# SOLUCION EJERCICIO 4: otro system_prompt, misma tool
agente_una_linea = create_agent(
    model=f"openai:{MODEL}",
    tools=[vuelos_tool],                       # la misma tool de siempre
    system_prompt="Contesta en una sola linea, sin listas.",
)
estado_corto = agente_una_linea.invoke(
    {"messages": [{"role": "user", "content": PREGUNTA_VUELOS}]}
)
print(estado_corto["messages"][-1].content)


### EJERCICIO 5: el grafo, dibujado por LangGraph

`draw_mermaid()` te imprime el grafo sin instalar nada. Buscá la ultima flecha:
`tools -.-> model`. Esa es el `while` que escribimos a mano.

Ojo con un detalle: la flecha es **punteada** (`-.->`), no solida. En Mermaid la
punteada marca una arista condicional, que es justo lo que era nuestro `if`.


In [ ]:
# SOLUCION EJERCICIO 5: imprimir el grafo
diagrama = agente_vuelos_lg.get_graph().draw_mermaid()
print(diagrama)

# La flecha que nos importa. Fijate que es punteada: es una arista condicional.
print("la flecha del loop esta?:", "tools -.-> model" in diagrama)


### EJERCICIO 6: que pasa sin `parse_docstring=True`

El mas interesante de los tres. Sin el flag, LangChain **no** adivina: te mete el
docstring entero, con el bloque `Args:` y todo, dentro de un solo campo `description`,
y las descripciones por argumento quedan en `None`.

El modelo lee esas descripciones por argumento para decidir que mandar. Perderlas no
rompe nada de forma visible, y esa es exactamente la clase de detalle que despues te
cuesta una tarde de debugging.


In [ ]:
# SOLUCION EJERCICIO 6: el mismo docstring, parseado mal a proposito
sin_flag = tool(buscar_vuelos)                    # sin parse_docstring=True
esquema_sin = convert_to_openai_tool(sin_flag)
esquema_con = convert_to_openai_tool(vuelos_tool)  # el de la celda de arriba

print("=== SIN parse_docstring=True ===")
print(json.dumps(esquema_sin, indent=2, ensure_ascii=False))
print()

# La diferencia que importa, lado a lado.
props_sin = esquema_sin["function"]["parameters"]["properties"]
props_con = esquema_con["function"]["parameters"]["properties"]
print("descripciones por argumento SIN el flag:",
      {k: v.get("description") for k, v in props_sin.items()})
print("descripciones por argumento CON el flag:",
      {k: v.get("description") for k, v in props_con.items()})


### EJERCICIO 7: arreglarlo escribiendo mejor los docstrings

No hace falta partir el agente para mejorar el ruteo: alcanza con que el limite entre las
gemelas sea explicito. Fijate que ahora cada docstring dice cuando **no** usarse.

Esto mejora mucho el ruteo, y sigue siendo fragil: depende de que cada persona que agregue
una tool escriba el limite bien. Por eso la solucion de fondo es la de la seccion que viene.


In [ ]:
# SOLUCION EJERCICIO 7: limites explicitos en los docstrings
def buscar_vuelos_baratos_v2(origen: str, destino: str) -> list[dict]:
    """Busca SOLO ofertas y promociones de ultimo momento en vuelos.

    No usar para una busqueda normal de vuelos: para eso usa buscar_vuelos.
    Usar unicamente si el usuario pide explicitamente ofertas o descuentos.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    return _normalizar(_serper(f"ofertas vuelos {origen} {destino}"))


def buscar_alojamiento_v2(ciudad: str, zona: str = "centro") -> list[dict]:
    """Busca SOLO hostels y departamentos temporarios, no hoteles.

    No usar para hoteles: para eso usa buscar_hoteles.

    Args:
        ciudad: por ejemplo "Bariloche"
        zona: barrio o area, por defecto "centro"
    """
    return _normalizar(_serper(f"hostel departamento temporario {zona} {ciudad}"))


agente_arreglado = AgenteVuelosMuchasTools(
    tools={
        "buscar_vuelos": buscar_vuelos,
        "buscar_vuelos_baratos": buscar_vuelos_baratos_v2,   # con el limite claro
        "buscar_hoteles": buscar_hoteles,
        "buscar_alojamiento": buscar_alojamiento_v2,         # con el limite claro
        "buscar_transporte": buscar_transporte,
        "buscar_actividades": buscar_actividades,
    },
    system=SYSTEM_CONFUNDIDO,     # el MISMO prompt de antes, no lo tocamos
)
agente_arreglado.run(PEDIDO)
print()
print("ESPERABAMOS:", sorted(TOOLS_ESPERADAS))
print("USO:        ", agente_arreglado.tools_usadas)


### EJERCICIO 8: por que bajar `temperature` no alcanza

Porque la temperatura cambia **cuanto** varia la eleccion, no **cual** es la mejor
candidata. Con las descripciones pisadas, la gemela es la opcion mas probable para el
modelo, y a temperatura 0 lo que hacemos es elegir la mas probable siempre: la equivocada,
pero de forma consistente.


In [ ]:
# SOLUCION EJERCICIO 8: temperature=0 no lo arregla
class AgenteFrio(AgenteVuelosMuchasTools):
    """El mismo agente confundido, pero con temperature=0."""

    def run(self, pregunta: str, max_turns: int = 5) -> str:
        # Reusamos todo el loop del padre y solo cambiamos la temperatura,
        # parcheando el cliente por un momento. Mas simple: copiar el metodo.
        self.messages.append({"role": "user", "content": pregunta})
        turno = 0
        while turno < max_turns:
            turno += 1
            resp = client.chat.completions.create(
                model=self.model,
                temperature=0.0,             # <-- el unico cambio
                messages=self.messages,
                tools=self._tool_schemas(),
            )
            msg = resp.choices[0].message
            if not msg.tool_calls:
                return msg.content
            self.messages.append(msg.model_dump(exclude_none=True))
            for call in msg.tool_calls:
                nombre = call.function.name
                args = json.loads(call.function.arguments)
                self.tools_usadas.append(nombre)
                resultado = self.tools[nombre](**args)
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(resultado[:3], ensure_ascii=False),
                })
        return "Me quede sin turnos."


agente_frio = AgenteFrio(tools=TOOLS_MUCHAS, system=SYSTEM_CONFUNDIDO)
agente_frio.run(PEDIDO)
print("USO con temperature=0:", agente_frio.tools_usadas)
# Las mismas gemelas. La temperatura no arregla un catalogo ambiguo:
# solo hace que elijas la opcion equivocada de forma mas consistente.


### EJERCICIO 9: el tercer especialista

Misma forma que los otros dos. Una capacidad, su tool, su limite. Si te sale igual que los
anteriores, entendiste el patron: es repetitivo a proposito.


In [ ]:
# SOLUCION EJERCICIO 9: agente_actividades_esp
tool_actividades = tool(buscar_actividades, parse_docstring=True)

agente_actividades_esp = create_agent(
    model=llm,
    tools=[tool_actividades],
    name="agente_actividades",
    system_prompt=(
        "Sos un especialista en actividades y paseos dentro de Argentina. "
        "Tu unica capacidad es buscar que hacer en una ciudad: si te piden vuelos "
        "u hoteles, decis que eso no es lo tuyo y no inventas. "
        "Contestas corto, en castellano rioplatense."
    ),
)

print(correr(agente_actividades_esp, "Que puedo hacer en Bariloche el fin de semana?"))


### EJERCICIO 10: dos tools alcanzan, si se pisan

El resultado es el que importa de todo el ejercicio: con **solo dos tools**, el agente
igual elige la gemela. No era un problema de cantidad. Era el limite.


In [ ]:
# SOLUCION EJERCICIO 10: un agente chico, dos tools que se pisan
agente_chico = AgenteVuelosMuchasTools(
    tools={
        "buscar_vuelos": buscar_vuelos,
        "buscar_vuelos_baratos": buscar_vuelos_baratos,   # la gemela
    },
    system="Sos un asistente de viajes. Priorizas siempre el mejor precio.",
)
agente_chico.run("Buscame vuelos de Buenos Aires a Bariloche")
print("con solo 2 tools eligio:", agente_chico.tools_usadas)
print("esperabamos: ['buscar_vuelos']")
# Dos tools. Igual elige la gemela. El problema nunca fue el numero.


### EJERCICIO 11: el handoff, escrito a mano

Nombrar al otro agente en el prompt es el handoff hecho a mano. Funciona como texto: el
agente le dice al usuario a quien preguntarle. Lo que **no** hace es pasarle el trabajo por
si mismo, y eso es justo lo que resuelven el supervisor y el swarm.


In [ ]:
# SOLUCION EJERCICIO 11: que cada uno nombre al otro
agente_vuelos_v2 = create_agent(
    model=llm,
    tools=[tool_vuelos],
    name="agente_vuelos",
    system_prompt=(
        "Sos un especialista en vuelos dentro de Argentina. "
        "Tu unica capacidad es buscar vuelos. "
        "Si te piden hoteles o alojamiento, aclara que eso lo maneja "
        "'agente_hoteles' y que vos no podes hacerlo. "
        "Contestas corto, en castellano rioplatense."
    ),
)

print(correr(agente_vuelos_v2, "Necesito un hotel en Bariloche"))
# Ahora no solo dice que no es lo suyo: dice a quien hay que preguntarle.
# Sigue siendo texto, no un pase de verdad. Eso viene despues.


### EJERCICIO 12: un supervisor mas callado

El prompt del supervisor es la unica palanca. Los especialistas siguen devolviendo sus
listas largas: lo que cambia es quien resume al final.


In [ ]:
# SOLUCION EJERCICIO 12: el supervisor contesta corto
PROMPT_SUPERVISOR_CORTO = (
    "Sos el coordinador de un concierge de viajes en Argentina.\n"
    "Tenes dos especialistas: agente_vuelos para vuelos y agente_hoteles para hoteles.\n"
    "Derivales de a uno lo que corresponda.\n"
    "No inventes datos: usa solo lo que te pasaron los especialistas.\n"
    # La linea nueva:
    "Contesta en UN parrafo corto, sin listas y sin links, como un mensaje de "
    "WhatsApp a un amigo. Menciona un precio concreto de cada cosa."
)

supervisor_corto = create_supervisor(
    [agente_vuelos_esp, agente_hoteles_esp],
    model=llm,
    prompt=PROMPT_SUPERVISOR_CORTO,
    output_mode="full_history",
    handoff_tool_prefix="derivar_a_",
).compile()

salida = supervisor_corto.invoke(
    {"messages": [{"role": "user", "content": PEDIDO_BARILOCHE}]}
)
print(salida["messages"][-1].content)


### EJERCICIO 13: el tercer especialista, adentro del supervisor

Dos cambios: el agente en la lista, y su nombre en el prompt. Si agregas el agente pero no
lo nombras en el prompt, el supervisor no sabe que existe y nunca le deriva.


In [ ]:
# SOLUCION EJERCICIO 13: sumamos actividades al equipo
tool_actividades_sup = tool(buscar_actividades, parse_docstring=True)

agente_actividades_sup = create_agent(
    model=llm,
    tools=[tool_actividades_sup],
    name="agente_actividades",
    system_prompt=(
        "Sos un especialista en actividades y paseos dentro de Argentina. "
        "Tu unica capacidad es buscar que hacer en una ciudad. "
        "Contestas corto, en castellano rioplatense."
    ),
)

PROMPT_SUPERVISOR_TRES = (
    "Sos el coordinador de un concierge de viajes en Argentina.\n"
    # Los TRES nombres, si no el supervisor no sabe que el tercero existe.
    "Tenes tres especialistas: agente_vuelos para vuelos, agente_hoteles para "
    "hoteles y agente_actividades para que hacer.\n"
    "Derivales de a uno lo que corresponda.\n"
    "No inventes datos: usa solo lo que te pasaron los especialistas.\n"
    "Cuando tengas todas las respuestas, escribi vos la respuesta final, "
    "en un solo mensaje."
)

supervisor_tres = create_supervisor(
    [agente_vuelos_esp, agente_hoteles_esp, agente_actividades_sup],
    model=llm,
    prompt=PROMPT_SUPERVISOR_TRES,
    output_mode="full_history",
    handoff_tool_prefix="derivar_a_",
).compile()

PEDIDO_COMPLETO = (
    "Buscame vuelos a Bariloche desde Buenos Aires, un hotel cerca del centro "
    "y que hacer el finde."
)

for paso in supervisor_tres.stream(
    {"messages": [{"role": "user", "content": PEDIDO_COMPLETO}]}
):
    for nodo in paso:
        print("  ->", nodo)


### EJERCICIO 14: el supervisor inventando

Sin la linea "No inventes datos", el supervisor contesta sobre el Cerro Catedral y el
Circuito Chico **sin haber llamado ninguna tool**. Sabe de Bariloche porque lo leyo en su
entrenamiento, y lo sirve con la misma seguridad que los datos que si fue a buscar.

Esa es la razon de ser de esa linea, y el riesgo real de darle la ultima palabra a un
modelo: no distingue entre lo que averiguo hoy y lo que recuerda.


In [ ]:
# SOLUCION EJERCICIO 14: sin el freno, invents
PROMPT_SIN_FRENO = (
    "Sos el coordinador de un concierge de viajes en Argentina.\n"
    "Tenes dos especialistas: agente_vuelos para vuelos y agente_hoteles para hoteles.\n"
    "Derivales de a uno lo que corresponda.\n"
    # Saque la linea "No inventes datos".
    "Cuando tengas las respuestas, escribi vos la respuesta final al usuario."
)

supervisor_suelto = create_supervisor(
    [agente_vuelos_esp, agente_hoteles_esp],   # SIN agente de actividades
    model=llm,
    prompt=PROMPT_SIN_FRENO,
    output_mode="full_history",
    handoff_tool_prefix="derivar_a_",
).compile()

pedido_tres_partes = (
    "Buscame vuelos a Bariloche desde Buenos Aires, un hotel cerca del centro "
    "y que hacer el finde."
)
resultado_suelto = supervisor_suelto.invoke(
    {"messages": [{"role": "user", "content": pedido_tres_partes}]}
)

print("tools que uso:", tools_que_uso(resultado_suelto))
print()
print(resultado_suelto["messages"][-1].content)
# Fijate: no hay ninguna tool de actividades en esa lista, y sin embargo la
# respuesta habla del finde en Bariloche. Eso se lo invento.


### EJERCICIO 15: una tool mas en el servidor

Lo que importa del ejercicio: **el cliente no cambia ni una linea.** Reescribis el archivo
del servidor, volves a pedir `list_tools()`, y la tool nueva aparece. Eso es lo que compra
el protocolo.


In [ ]:
# SOLUCION EJERCICIO 15: reescribimos el servidor con una tool mas
# (en el notebook esto va en la celda del %%writefile; aca lo escribimos con Python
# para que la solucion sea una sola celda que corre sola)
codigo_servidor = open("servidor_serper.py").read()

tool_nueva = '''

@servidor_serper.tool()
def buscar_actividades(ciudad: str) -> list[dict]:
    """Busca actividades y excursiones en una ciudad argentina.

    Args:
        ciudad: por ejemplo "Bariloche"
    """
    return _normalizar(_serper(f"que hacer en {ciudad} excursiones"))

'''

# La insertamos antes del bloque __main__, que tiene que quedar al final.
marca = 'if __name__ == "__main__":'
codigo_nuevo = codigo_servidor.replace(marca, tool_nueva + marca)
open("servidor_serper.py", "w").write(codigo_nuevo)

# Y volvemos a descubrir, con el MISMO cliente de antes.
adapter_v2 = MCPAdapter(Path("servidor_serper.py"))
tools_v2 = await adapter_v2.list_tools()
print("tools ahora:", [t.name for t in tools_v2])


### EJERCICIO 16: la trampa del `.env`, en vivo

El resultado es la leccion: **`list_tools()` anda perfecto** y el error aparece recien
cuando la tool se ejecuta. El servidor arranco, negocio el protocolo, listo sus tools. Todo
verde. Y explota una llamada despues.

Asi se ve un bug de entorno en un subproceso: tarde, y lejos de la causa.


In [ ]:
# SOLUCION EJERCICIO 16: sin load_dotenv(), falla tarde
roto = open("servidor_serper.py").read().replace("load_dotenv()", "# load_dotenv()")
open("servidor_serper_roto.py", "w").write(roto)

adapter_roto = MCPAdapter(Path("servidor_serper_roto.py"))

# Paso 1: descubrir las tools. Esto FUNCIONA, y ahi esta la trampa.
tools_roto = await adapter_roto.list_tools()
print("list_tools() anduvo igual:", [t.name for t in tools_roto])

# Paso 2: ejecutar una. ACA explota.
try:
    await tools_roto[0].ainvoke({"origen": "Buenos Aires", "destino": "Bariloche"})
except Exception as e:
    print()
    print("y recien al ejecutar la tool:")
    print(f"  {type(e).__name__}: {e}")
